# VARIAN CHIRPS
Identik dengan Eksperimen_Revisi_R1, tetapi TOTAL_HUJAN, BULAN_BASAH, dan data hujan bulanan (Curah-hujan.csv) berasal dari CHIRPS v2.0 (0,05°) per kecamatan (lihat ekstrak_chirps.py dan siapkan_data_chirps.py). Suhu, radiasi, dan kelembaban tetap NASA POWER. Data NASA asli: *_NASA.csv.

BASELINE XGBOOST

Perhitungan

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import xgboost as xgb
from scipy.stats import gmean

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error
)

# =========================================================
# LOAD SHP BANYUMAS
# =========================================================
print("1. Memproses Data Geospasial (SHP)...")

gdf_all = gpd.read_file(
    "ADMINISTRASIDESA_AR_25K.shp"
)

gdf_banyumas = (
    gdf_all[
        gdf_all['WADMKK']
        .str.contains('BANYUMAS', case=False)
    ]
    .dissolve(by='WADMKC')
    .reset_index()
)

gdf_banyumas_utm = gdf_banyumas.to_crs(epsg=32749)

# =========================================================
# LOAD DATASET MASTER
# =========================================================
print("2. Memproses Dataset Master...")

df_dataset = pd.read_csv(
    "Dataset_Master_TA_Padi_Spatial.csv"
)

# =========================================================
# DATA CLEANING
# =========================================================
print("3. Membersihkan Data...")

# Rapikan nama kolom
df_dataset.columns = [
    col.strip()
    for col in df_dataset.columns
]

# Rename kolom
df_dataset.rename(
    columns={'YEAR_fitur': 'YEAR'},
    inplace=True
)

# Rapikan nama kecamatan
df_dataset['WADMKC'] = (
    df_dataset['WADMKC']
    .str.title()
    .str.strip()
)

# Samakan nama kecamatan
df_dataset['WADMKC'] = (
    df_dataset['WADMKC']
    .replace({
        'Baturraden': 'Baturaden',
        'Tambak'    : 'Tampak'
    })
)

# Filter kecamatan valid
kecamatan_valid = gdf_banyumas['WADMKC'].unique()

df_dataset = df_dataset[
    df_dataset['WADMKC'].isin(kecamatan_valid)
]

# Sort data
df_dataset = df_dataset.sort_values(
    ['WADMKC', 'YEAR']
).reset_index(drop=True)

# =========================================================
# MISSING VALUE HANDLING
# =========================================================
print("4. Handling Missing Value...")

numeric_cols = [
    'TOTAL_HUJAN',
    'BULAN_BASAH',
    'AVG_RADIASI',
    'AVG_KELEMBABAN',
    'MAX_SUHU_THN',
    'MIN_SUHU_THN',
    'AVG_SUHU',
    'LUAS_PANEN',
    'PRODUKSI_PADI',
    'PRODUKTIVITAS'
]

print("\nJumlah Missing Value Sebelum:")
print(df_dataset[numeric_cols].isnull().sum().to_string())

for col in numeric_cols:
    if df_dataset[col].isnull().sum() > 0:
        df_dataset[col] = df_dataset.groupby(
            'WADMKC'
        )[col].transform(
            lambda x: x.fillna(x.median())
        )
        df_dataset[col] = df_dataset[col].fillna(
            df_dataset[col].median()
        )

print("\nJumlah Missing Value Sesudah:")
print(df_dataset[numeric_cols].isnull().sum().to_string())

# =========================================================
# STEP 4.5 : ANALISIS DATA EKSPLORATIF (EDA)
# =========================================================
print("\n4.5. Analisis Data Eksploratif (EDA)...")

# 1. Menampilkan Statistik Deskriptif (Max, Min, Mean)
print("\nStatistik Deskriptif Dataset:")
print(df_dataset[numeric_cols].describe().round(2).to_string())

# =========================================================
# DETEKSI OUTLIER
# =========================================================
print("5. Deteksi Outlier (IQR Method)...")

for col in numeric_cols:
    Q1 = df_dataset[col].quantile(0.25)
    Q3 = df_dataset[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - (1.5 * IQR)
    upper = Q3 + (1.5 * IQR)
    outlier_count = len(
        df_dataset[
            (df_dataset[col] < lower) |
            (df_dataset[col] > upper)
        ]
    )
    print(
        f"{col:<25}: {outlier_count} outlier "
        f"[LB={lower:.2f}, UB={upper:.2f}]"
    )

print("Outlier dipertahankan")

# =========================================================
# PERSIAPAN FITUR
# =========================================================
print("6. Menyiapkan Fitur Agroklimat...")

X_COLS = [
    'TOTAL_HUJAN',
    'BULAN_BASAH',
    'AVG_RADIASI',
    'AVG_KELEMBABAN',
    'MAX_SUHU_THN',
    'MIN_SUHU_THN',
    'AVG_SUHU'
]

df_agroklimat = df_dataset[
    ['WADMKC', 'YEAR'] + X_COLS
].copy()

# =========================================================
# LAGGING & PERHITUNGAN RASIO LUAS PANEN (GEOMETRIC MEAN)
# =========================================================
print("7. Menerapkan Teknik Lagging & Rasio Luas Panen (Geometric Mean)...")

df_dataset['LUAS_PANEN_PREV'] = df_dataset.groupby(
    'WADMKC'
)['LUAS_PANEN'].shift(1)

df_dataset['RASIO_LUAS'] = (
    df_dataset['LUAS_PANEN']
    / df_dataset['LUAS_PANEN_PREV']
)

df_rasio_recent = df_dataset[
    df_dataset['YEAR'].isin([2022, 2023, 2024])
].dropna(subset=['RASIO_LUAS'])

# Menghitung Geometric Mean untuk rasio perubahan luas panen (menghilangkan upward bias aritmatika)
FAKTOR_PER_KEC = (
    df_rasio_recent
    .groupby('WADMKC')['RASIO_LUAS']
    .apply(gmean)
)

# Standar deviasi fluktuasi rasio per kecamatan untuk propagasi ketidakpastian
STD_RATIO_PER_KEC = (
    df_rasio_recent
    .groupby('WADMKC')['RASIO_LUAS']
    .std()
    .fillna(0.05)
)

FAKTOR_GLOBAL = gmean(FAKTOR_PER_KEC)

print(f"Faktor Pertumbuhan Luas (Geometric Mean Global): {FAKTOR_GLOBAL:.4f}")

# =========================================================
# DATA PREDIKSI 2025
# =========================================================
df_input_2025 = df_dataset[
    df_dataset['YEAR'] == 2024
].copy()

df_input_2025['FAKTOR_KOREKSI'] = (
    df_input_2025['WADMKC']
    .map(FAKTOR_PER_KEC)
    .fillna(FAKTOR_GLOBAL)
)

df_input_2025['STD_RASIO'] = (
    df_input_2025['WADMKC']
    .map(STD_RATIO_PER_KEC)
    .fillna(0.05)
)

df_input_2025['LUAS_PANEN_2025_EST'] = (
    df_input_2025['LUAS_PANEN']
    * df_input_2025['FAKTOR_KOREKSI']
)

# =========================================================
# BUILD DATASET LAGGING
# =========================================================
df_fitur_latih = df_agroklimat.copy()

df_fitur_latih['YEAR_TARGET'] = (
    df_fitur_latih['YEAR'] + 1
)

df_target = df_dataset[
    [
        'WADMKC',
        'YEAR',
        'PRODUKSI_PADI',
        'LUAS_PANEN',
        'PRODUKTIVITAS'
    ]
].copy()

df_master = pd.merge(
    df_fitur_latih,
    df_target,
    left_on=['WADMKC', 'YEAR_TARGET'],
    right_on=['WADMKC', 'YEAR'],
    how='inner',
    suffixes=('_fitur', '_target')
)

print(f"Jumlah data training: {len(df_master)}")

# Produktivitas tahun fitur (t): dipakai untuk baseline persistence & model anomali
prod_t = df_dataset.set_index(['WADMKC', 'YEAR'])['PRODUKTIVITAS']
df_master['PRODUKTIVITAS_T'] = [
    prod_t[(kec, yr)] for kec, yr in zip(df_master['WADMKC'], df_master['YEAR_fitur'])
]
df_master['DELTA_PRODUKTIVITAS'] = df_master['PRODUKTIVITAS'] - df_master['PRODUKTIVITAS_T']

# =========================================================
# PEMODELAN XGBOOST (TRUE SLIDING WINDOW & EXPANDING COMP.)
# =========================================================
print("8. Pemodelan XGBoost dengan Komparasi Lengkap (True Sliding Window)...")

X_raw = df_master[X_COLS + ['WADMKC', 'YEAR_fitur']]
X_processed = pd.get_dummies(X_raw, columns=['WADMKC'], dtype=int)
X_processed.rename(columns={'YEAR_fitur': 'YEAR_INDEX'}, inplace=True)
y = df_master['PRODUKTIVITAS']
max_year = df_master['YEAR_fitur'].max()
min_available_year = df_master['YEAR_fitur'].min()
years_available = sorted(df_master['YEAR_fitur'].unique())

eval_results_produktivitas = []
eval_results_produksi = []

model_terbaik = None
r2_terbaik = -999
rasio_terbaik = ""

# 1. Evaluasi Expanding Window Skenario
SCENARIOS_FIXED = {
    f'Expanding (Train <= {max_year-3})': {
        'train_query': df_master['YEAR_fitur'] <= (max_year - 3),
        'test_query': df_master['YEAR_fitur'] > (max_year - 3)
    },
    f'Expanding (Train <= {max_year-2})': {
        'train_query': df_master['YEAR_fitur'] <= (max_year - 2),
        'test_query': df_master['YEAR_fitur'] > (max_year - 2)
    },
    f'Expanding (Train <= {max_year-1})': {
        'train_query': df_master['YEAR_fitur'] <= (max_year - 1),
        'test_query': df_master['YEAR_fitur'] > (max_year - 1)
    }
}

for label, queries in SCENARIOS_FIXED.items():
    train_idx = df_master[queries['train_query']].index
    test_idx = df_master[queries['test_query']].index
    if len(train_idx) == 0 or len(test_idx) == 0: continue

    X_train, X_test = X_processed.loc[train_idx], X_processed.loc[test_idx]
    y_train, y_test = y.loc[train_idx], y.loc[test_idx]

    model = xgb.XGBRegressor(random_state=42)
    model.fit(X_train, y_train)
    y_pred_prod = model.predict(X_test)

    luas_panen_test = df_master.loc[test_idx, 'LUAS_PANEN']
    y_test_ton = y_test * luas_panen_test
    y_pred_ton = y_pred_prod * luas_panen_test

    # Metrik PRODUKTIVITAS (ton/ha)
    r2_v = r2_score(y_test, y_pred_prod)
    rmse_v = np.sqrt(mean_squared_error(y_test, y_pred_prod))
    mae_v = mean_absolute_error(y_test, y_pred_prod)
    mape_v = mean_absolute_percentage_error(y_test, y_pred_prod) * 100

    # Metrik PRODUKSI (ton)
    r2_p = r2_score(y_test_ton, y_pred_ton)
    rmse_p = np.sqrt(mean_squared_error(y_test_ton, y_pred_ton))
    mae_p = mean_absolute_error(y_test_ton, y_pred_ton)
    mape_p = mean_absolute_percentage_error(y_test_ton, y_pred_ton) * 100

    eval_results_produktivitas.append({
        'Skenario Split': label,
        'Data Latih (N)': len(X_train),
        'Data Uji (N)': len(X_test),
        'R²': round(r2_v, 4),
        'RMSE (Ton/Ha)': round(rmse_v, 4),
        'MAE (Ton/Ha)': round(mae_v, 4),
        'MAPE (%)': round(mape_v, 2)
    })

    eval_results_produksi.append({
        'Skenario Split': label,
        'Data Latih (N)': len(X_train),
        'Data Uji (N)': len(X_test),
        'R²': round(r2_p, 4),
        'RMSE (Ton)': round(rmse_p, 2),
        'MAE (Ton)': round(mae_p, 2),
        'MAPE (%)': round(mape_p, 2)
    })

    if r2_v > r2_terbaik:
        r2_terbaik = r2_v
        model_terbaik = model
        rasio_terbaik = label

# 2. Evaluasi True Sliding Window (Walk-Forward Validation) untuk beberapa Window Size
for w in [5, 6, 7, 8, 9, 10]:
    if len(years_available) > w:
        temp_r2_v, temp_rmse_v, temp_mae_v, temp_mape_v = [], [], [], []
        temp_r2_p, temp_rmse_p, temp_mae_p, temp_mape_p = [], [], [], []
        total_train_size, total_test_size = 0, 0
        step_count = 0

        # Geser jendela maju satu tahun secara berulang
        for start_idx in range(len(years_available) - w):
            train_years = years_available[start_idx : start_idx + w]
            test_year = years_available[start_idx + w]

            t_idx = df_master[df_master['YEAR_fitur'].isin(train_years)].index
            v_idx = df_master[df_master['YEAR_fitur'] == test_year].index

            if len(t_idx) == 0 or len(v_idx) == 0: continue

            X_tr, X_val = X_processed.loc[t_idx], X_processed.loc[v_idx]
            y_tr, y_val = y.loc[t_idx], y.loc[v_idx]

            m_fold = xgb.XGBRegressor(random_state=42)
            m_fold.fit(X_tr, y_tr)
            y_pred_prod = m_fold.predict(X_val)

            luas_val = df_master.loc[v_idx, 'LUAS_PANEN']
            y_val_ton = y_val * luas_val
            y_pred_ton = y_pred_prod * luas_val

            # Produktivitas (ton/ha)
            temp_r2_v.append(r2_score(y_val, y_pred_prod))
            temp_rmse_v.append(np.sqrt(mean_squared_error(y_val, y_pred_prod)))
            temp_mae_v.append(mean_absolute_error(y_val, y_pred_prod))
            temp_mape_v.append(mean_absolute_percentage_error(y_val, y_pred_prod) * 100)

            # Produksi (ton)
            temp_r2_p.append(r2_score(y_val_ton, y_pred_ton))
            temp_rmse_p.append(np.sqrt(mean_squared_error(y_val_ton, y_pred_ton)))
            temp_mae_p.append(mean_absolute_error(y_val_ton, y_pred_ton))
            temp_mape_p.append(mean_absolute_percentage_error(y_val_ton, y_pred_ton) * 100)

            total_train_size += len(X_tr)
            total_test_size += len(X_val)
            step_count += 1

        if step_count > 0:
            r2_avg_v = np.mean(temp_r2_v)
            r2_avg_p = np.mean(temp_r2_p)

            eval_results_produktivitas.append({
                'Skenario Split': f'True Sliding Window ({w} Thn - Rata-rata)',
                'Data Latih (N)': int(total_train_size / step_count),
                'Data Uji (N)': int(total_test_size / step_count),
                'R²': round(r2_avg_v, 4),
                'RMSE (Ton/Ha)': round(np.mean(temp_rmse_v), 4),
                'MAE (Ton/Ha)': round(np.mean(temp_mae_v), 4),
                'MAPE (%)': round(np.mean(temp_mape_v), 2)
            })

            eval_results_produksi.append({
                'Skenario Split': f'True Sliding Window ({w} Thn - Rata-rata)',
                'Data Latih (N)': int(total_train_size / step_count),
                'Data Uji (N)': int(total_test_size / step_count),
                'R²': round(r2_avg_p, 4),
                'RMSE (Ton)': round(np.mean(temp_rmse_p), 2),
                'MAE (Ton)': round(np.mean(temp_mae_p), 2),
                'MAPE (%)': round(np.mean(temp_mape_p), 2)
            })

            if r2_avg_v > r2_terbaik:
                r2_terbaik = r2_avg_v
                last_train_years = years_available[-w:]
                last_train_idx = df_master[
                    df_master["YEAR_fitur"].isin(last_train_years)
                ].index

                model_terbaik = xgb.XGBRegressor(random_state=42)
                model_terbaik.fit(
                    X_processed.loc[last_train_idx], y.loc[last_train_idx]
                )
                rasio_terbaik = f"True Sliding Window ({w} Thn: {min(last_train_years)}-{max(last_train_years)})"

                print(
                    f"Model final dilatih menggunakan data fitur tahun: {min(last_train_years)} s.d. {max(last_train_years)}"
                )
                print(f"Jumlah sampel latih final: {len(last_train_idx)} baris.")
    else:
        print(f"Catatan: Window {w} tahun tidak dapat diuji dengan True Sliding karena data tidak cukup.")

# 3. Tampilkan DataFrame Komparasi
df_komparasi_produktivitas = pd.DataFrame(eval_results_produktivitas)
df_komparasi_produksi = pd.DataFrame(eval_results_produksi)

print("\n" + "=" * 70)
print("TABEL 1. KOMPARASI PERFORMA EVALUASI PRODUKTIVITAS (TON/HA) - TARGET UTAMA")
print("=" * 70)
display(df_komparasi_produktivitas)

print("\n" + "=" * 70)
print("TABEL 2. KOMPARASI PERFORMA EVALUASI PRODUKSI (TON) - HASIL KONVERSI")
print("=" * 70)
display(df_komparasi_produksi)

print(f"\n-> Model Terbaik Terpilih: {rasio_terbaik} dengan R² (Produktivitas) = {r2_terbaik:.4f}")


1. Memproses Data Geospasial (SHP)...


2. Memproses Dataset Master...
3. Membersihkan Data...
4. Handling Missing Value...

Jumlah Missing Value Sebelum:
TOTAL_HUJAN       0
BULAN_BASAH       0
AVG_RADIASI       0
AVG_KELEMBABAN    0
MAX_SUHU_THN      0
MIN_SUHU_THN      0
AVG_SUHU          0
LUAS_PANEN        0
PRODUKSI_PADI     0
PRODUKTIVITAS     0

Jumlah Missing Value Sesudah:
TOTAL_HUJAN       0
BULAN_BASAH       0
AVG_RADIASI       0
AVG_KELEMBABAN    0
MAX_SUHU_THN      0
MIN_SUHU_THN      0
AVG_SUHU          0
LUAS_PANEN        0
PRODUKSI_PADI     0
PRODUKTIVITAS     0

4.5. Analisis Data Eksploratif (EDA)...

Statistik Deskriptif Dataset:
       TOTAL_HUJAN  BULAN_BASAH  AVG_RADIASI  AVG_KELEMBABAN  MAX_SUHU_THN  MIN_SUHU_THN  AVG_SUHU  LUAS_PANEN  PRODUKSI_PADI  PRODUKTIVITAS
count       297.00       297.00       297.00          297.00        297.00        297.00    297.00      297.00         297.00         297.00
mean       3459.78         7.75        17.83           86.78         33.07         17.65     25.28  

C:\Users\acer\AppData\Local\Temp\ipykernel_16572\2377119636.py:56: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_dataset.rename(


Faktor Pertumbuhan Luas (Geometric Mean Global): 0.9509
Jumlah data training: 270
8. Pemodelan XGBoost dengan Komparasi Lengkap (True Sliding Window)...


C:\Users\acer\AppData\Local\Temp\ipykernel_16572\2377119636.py:281: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  X_processed.rename(columns={'YEAR_fitur': 'YEAR_INDEX'}, inplace=True)


Model final dilatih menggunakan data fitur tahun: 2017 s.d. 2023
Jumlah sampel latih final: 189 baris.


Catatan: Window 10 tahun tidak dapat diuji dengan True Sliding karena data tidak cukup.

TABEL 1. KOMPARASI PERFORMA EVALUASI PRODUKTIVITAS (TON/HA) - TARGET UTAMA


,Skenario Split,Data Latih (N),Data Uji (N),R²,RMSE (Ton/Ha),MAE (Ton/Ha),MAPE (%)
0,Expanding (Train <= 2020),189,81,0.1149,0.3892,0.2831,4.85
1,Expanding (Train <= 2021),216,54,0.2591,0.3536,0.2563,4.38
2,Expanding (Train <= 2022),243,27,0.3136,0.3399,0.2222,3.90
3,True Sliding Window (5 Thn - Rata-rata),135,27,0.1012,0.3513,0.2631,4.54
4,True Sliding Window (6 Thn - Rata-rata),162,27,-0.5791,0.4359,0.3412,5.83
5,True Sliding Window (7 Thn - Rata-rata),189,27,0.3594,0.3305,0.2372,4.07
6,True Sliding Window (8 Thn - Rata-rata),216,27,0.2424,0.3562,0.2620,4.46
7,True Sliding Window (9 Thn - Rata-rata),243,27,0.3136,0.3399,0.2222,3.90



TABEL 2. KOMPARASI PERFORMA EVALUASI PRODUKSI (TON) - HASIL KONVERSI


,Skenario Split,Data Latih (N),Data Uji (N),R²,RMSE (Ton),MAE (Ton),MAPE (%)
0,Expanding (Train <= 2020),189,81,0.9766,1071.28,692.76,4.85
1,Expanding (Train <= 2021),216,54,0.9802,1003.84,627.48,4.38
2,Expanding (Train <= 2022),243,27,0.9843,879.53,487.81,3.90
3,True Sliding Window (5 Thn - Rata-rata),135,27,0.9817,929.00,630.52,4.54
4,True Sliding Window (6 Thn - Rata-rata),162,27,0.9698,1180.73,827.77,5.83
5,True Sliding Window (7 Thn - Rata-rata),189,27,0.9840,883.26,572.30,4.07
6,True Sliding Window (8 Thn - Rata-rata),216,27,0.9791,1017.82,651.23,4.46
7,True Sliding Window (9 Thn - Rata-rata),243,27,0.9843,879.53,487.81,3.90



-> Model Terbaik Terpilih: True Sliding Window (7 Thn: 2017-2023) dengan R² (Produktivitas) = 0.3594


## DIAGNOSTIK: RESOLUSI SPASIAL NASA POWER & KUALITAS DATA
Menjawab Reviewer 1 poin 3: jumlah sel grid unik yang melayani 27 kecamatan, jumlah nilai unik fitur iklim per tahun, serta pemeriksaan kualitas data produktivitas per tahun.

In [2]:
# =========================================================
# DIAGNOSTIK RESOLUSI SPASIAL NASA POWER & KUALITAS DATA
# Menjawab Reviewer 1 poin 3 (jumlah sel grid unik) & Reviewer 2 (keterbatasan data)
# =========================================================
import numpy as np
import pandas as pd
import geopandas as gpd
from IPython.display import display

print("=== DIAGNOSTIK RESOLUSI SPASIAL & KUALITAS DATA ===")

# ---------------------------------------------------------
# 1. Pemetaan centroid kecamatan -> titik grid MERRA-2 terdekat (Persamaan 1)
# ---------------------------------------------------------
df_ch_grid = pd.read_csv("Curah-hujan.csv")
pts_grid = df_ch_grid[['LAT', 'LON']].drop_duplicates()
gdf_grid = gpd.GeoDataFrame(
    pts_grid,
    geometry=gpd.points_from_xy(pts_grid['LON'], pts_grid['LAT']),
    crs="EPSG:4326"
).to_crs(epsg=32749)

gdf_centroid = gdf_banyumas_utm[['WADMKC', 'geometry']].copy()
gdf_centroid['geometry'] = gdf_centroid.geometry.centroid

grid_map = (
    gpd.sjoin_nearest(gdf_centroid, gdf_grid, how='left')
    [['WADMKC', 'LAT', 'LON']]
    .drop_duplicates(subset=['WADMKC'])
)
df_grid_count = (
    grid_map.groupby(['LAT', 'LON'])['WADMKC']
    .agg(['count', lambda s: ', '.join(sorted(s))])
    .reset_index()
    .rename(columns={'count': 'Jumlah Kecamatan', '<lambda_0>': 'Kecamatan'})
)
N_GRID_MERRA = len(df_grid_count)

print("\nTABEL 1. TITIK GRID MERRA-2 (CURAH HUJAN) YANG MELAYANI 27 KECAMATAN")
display(df_grid_count)

# ---------------------------------------------------------
# 2. Jumlah nilai unik setiap fitur iklim per tahun (bukti langsung dari dataset)
#    AVG_RADIASI berasal dari CERES, sisanya MERRA-2.
# ---------------------------------------------------------
uniq_rows = []
for col in X_COLS:
    n_uniq = df_dataset.groupby('YEAR')[col].nunique()
    uniq_rows.append({
        'Fitur': col,
        'Sumber': 'CERES' if col == 'AVG_RADIASI' else 'MERRA-2',
        'Nilai Unik/Tahun (min)': int(n_uniq.min()),
        'Nilai Unik/Tahun (maks)': int(n_uniq.max()),
        'Tahun dgn 1 Nilai Untuk Semua Kecamatan': int((n_uniq == 1).sum()),
        'Total Tahun': int(len(n_uniq))
    })
df_uniq = pd.DataFrame(uniq_rows)

print("\nTABEL 2. JUMLAH NILAI UNIK FITUR IKLIM PER TAHUN (DARI 27 KECAMATAN)")
display(df_uniq)

max_uniq = df_uniq['Nilai Unik/Tahun (maks)'].max()
bb_row = df_uniq[df_uniq['Fitur'] == 'BULAN_BASAH'].iloc[0]

# ---------------------------------------------------------
# 3. Kualitas data target: sebaran produktivitas antar-kecamatan per tahun
# ---------------------------------------------------------
df_year_quality = (
    df_dataset.groupby('YEAR')['PRODUKTIVITAS']
    .agg(['mean', 'std', 'min', 'max'])
    .round(3)
)
df_year_quality['Perubahan Rata2 vs Tahun Lalu'] = (
    df_dataset.sort_values(['WADMKC', 'YEAR'])
    .assign(DELTA=lambda d: d.groupby('WADMKC')['PRODUKTIVITAS'].diff())
    .groupby('YEAR')['DELTA'].apply(lambda s: s.abs().mean())
    .round(3)
)
median_std = df_year_quality['std'].median()
suspect_years = df_year_quality[df_year_quality['std'] < 0.25 * median_std].index.tolist()

print("\nTABEL 3. SEBARAN PRODUKTIVITAS ANTAR-KECAMATAN PER TAHUN (TON/HA)")
display(df_year_quality)

print("\nRingkasan Diagnostik (dihitung otomatis):")
print(f"1. Centroid 27 kecamatan terpetakan ke {N_GRID_MERRA} titik grid MERRA-2 unik.")
print(f"2. Setiap fitur iklim hanya memiliki maksimal {max_uniq} nilai unik per tahun dari 27 kecamatan.")
print(f"   BULAN_BASAH bernilai sama untuk seluruh kecamatan pada "
      f"{bb_row['Tahun dgn 1 Nilai Untuk Semua Kecamatan']} dari {bb_row['Total Tahun']} tahun,")
print("   sehingga fitur iklim praktis merupakan sinyal tingkat TAHUN/REGIONAL, bukan tingkat kecamatan.")
if suspect_years:
    print(f"3. PERINGATAN kualitas data: tahun {suspect_years} memiliki simpangan baku produktivitas antar-kecamatan")
    print(f"   < 25% dari median ({median_std:.3f} ton/ha). Periksa apakah nilai tersebut merupakan angka")
    print("   tingkat kabupaten yang disalin ke seluruh kecamatan.")


=== DIAGNOSTIK RESOLUSI SPASIAL & KUALITAS DATA ===

TABEL 1. TITIK GRID MERRA-2 (CURAH HUJAN) YANG MELAYANI 27 KECAMATAN


,LAT,LON,Jumlah Kecamatan,Kecamatan
0,-7.60078,109.36138,1,Sumpiuh
1,-7.59256,109.41325,1,Tampak
2,-7.59126,109.30393,1,Kemranjen
3,-7.54154,109.21728,1,Kebasen
4,-7.53986,109.10849,1,Jatilawang
5,-7.53018,109.17082,1,Rawalo
6,-7.52903,109.27551,1,Banyumas
7,-7.51782,109.34926,1,Somagede
8,-7.51266,109.05529,1,Wangon
9,-7.48831,109.11418,1,Purwojati



TABEL 2. JUMLAH NILAI UNIK FITUR IKLIM PER TAHUN (DARI 27 KECAMATAN)


,Fitur,Sumber,Nilai Unik/Tahun (min),Nilai Unik/Tahun (maks),Tahun dgn 1 Nilai Untuk Semua Kecamatan,Total Tahun
0,TOTAL_HUJAN,MERRA-2,27,27,0,11
1,BULAN_BASAH,MERRA-2,1,3,2,11
2,AVG_RADIASI,CERES,2,2,0,11
3,AVG_KELEMBABAN,MERRA-2,2,2,0,11
4,MAX_SUHU_THN,MERRA-2,2,2,0,11
5,MIN_SUHU_THN,MERRA-2,2,2,0,11
6,AVG_SUHU,MERRA-2,2,2,0,11



TABEL 3. SEBARAN PRODUKTIVITAS ANTAR-KECAMATAN PER TAHUN (TON/HA)


,mean,std,min,max,Perubahan Rata2 vs Tahun Lalu
YEAR,,,,,
2014,4.964,0.069,4.828,5.091,NaN
2015,5.696,0.005,5.690,5.701,0.732
2016,5.581,0.500,4.201,6.101,0.401
2017,5.571,0.562,4.200,6.861,0.548
2018,5.818,0.480,4.625,6.663,0.472
2019,5.723,0.486,4.625,6.726,0.138
2020,5.806,0.420,4.919,6.719,0.155
2021,5.837,0.290,5.051,6.354,0.121
2022,5.756,0.425,4.793,6.599,0.264



Ringkasan Diagnostik (dihitung otomatis):
1. Centroid 27 kecamatan terpetakan ke 27 titik grid MERRA-2 unik.
2. Setiap fitur iklim hanya memiliki maksimal 27 nilai unik per tahun dari 27 kecamatan.
   BULAN_BASAH bernilai sama untuk seluruh kecamatan pada 2 dari 11 tahun,
   sehingga fitur iklim praktis merupakan sinyal tingkat TAHUN/REGIONAL, bukan tingkat kecamatan.
3. PERINGATAN kualitas data: tahun [2014, 2015] memiliki simpangan baku produktivitas antar-kecamatan
   < 25% dari median (0.420 ton/ha). Periksa apakah nilai tersebut merupakan angka
   tingkat kabupaten yang disalin ke seluruh kecamatan.


In [3]:
# =========================================================
# PENGUJIAN KOMPARASI FITUR 'BULAN_BASAH' (ABLATION TEST)
# Dilengkapi Multi-Seed (Mean ± SD), Lintas Skenario, & Permutation Importance
# =========================================================

import re
import numpy as np
import pandas as pd
import xgboost as xgb
from scipy import stats
from IPython.display import display
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)

print("=== PENGUJIAN KOMPARASI FITUR 'BULAN_BASAH' (ABLATION TEST) ===")

if "True Sliding Window" in rasio_terbaik:
    w = int(re.search(r"\d+", rasio_terbaik).group())
else:
    w = 8

SEEDS = [42, 10, 77, 123, 2024, 999, 55, 314, 7, 888]

# 1. Fungsi Evaluasi Walk-Forward Sliding Window
def evaluate_sliding_window(feature_cols, window_size=w, random_state=42, subsample=1.0, colsample_bytree=1.0):
    temp_r2_v, temp_rmse_v, temp_mae_v, temp_mape_v = [], [], [], []
    temp_r2_p, temp_rmse_p, temp_mae_p, temp_mape_p = [], [], [], []

    for start_idx in range(len(years_available) - window_size):
        train_years = years_available[start_idx : start_idx + window_size]
        test_year = years_available[start_idx + window_size]

        t_idx = df_master[df_master["YEAR_fitur"].isin(train_years)].index
        v_idx = df_master[df_master["YEAR_fitur"] == test_year].index

        if len(t_idx) == 0 or len(v_idx) == 0:
            continue

        X_tr = X_processed.loc[t_idx, feature_cols]
        X_val = X_processed.loc[v_idx, feature_cols]
        y_tr, y_val = y.loc[t_idx], y.loc[v_idx]

        model = xgb.XGBRegressor(
            random_state=random_state,
            subsample=subsample,
            colsample_bytree=colsample_bytree
        )
        model.fit(X_tr, y_tr)

        pred_prod = model.predict(X_val)
        luas_val = df_master.loc[v_idx, "LUAS_PANEN"]

        y_val_ton = y_val * luas_val
        pred_ton = pred_prod * luas_val

        temp_r2_v.append(r2_score(y_val, pred_prod))
        temp_rmse_v.append(np.sqrt(mean_squared_error(y_val, pred_prod)))
        temp_mae_v.append(mean_absolute_error(y_val, pred_prod))
        temp_mape_v.append(mean_absolute_percentage_error(y_val, pred_prod) * 100)

        temp_r2_p.append(r2_score(y_val_ton, pred_ton))
        temp_rmse_p.append(np.sqrt(mean_squared_error(y_val_ton, pred_ton)))
        temp_mae_p.append(mean_absolute_error(y_val_ton, pred_ton))
        temp_mape_p.append(mean_absolute_percentage_error(y_val_ton, pred_ton) * 100)

    return {
        "r2_v": np.mean(temp_r2_v),
        "rmse_v": np.mean(temp_rmse_v),
        "mae_v": np.mean(temp_mae_v),
        "mape_v": np.mean(temp_mape_v),
        "r2_p": np.mean(temp_r2_p),
        "rmse_p": np.mean(temp_rmse_p),
        "mae_p": np.mean(temp_mae_p),
        "mape_p": np.mean(temp_mape_p),
    }

# Evaluasi Expanding Window (train <= cutoff, uji seluruh tahun setelahnya)
def evaluate_expanding(feature_cols, cutoff, random_state=42, subsample=1.0, colsample_bytree=1.0):
    t_idx = df_master[df_master['YEAR_fitur'] <= cutoff].index
    v_idx = df_master[df_master['YEAR_fitur'] > cutoff].index
    model = xgb.XGBRegressor(
        random_state=random_state,
        subsample=subsample,
        colsample_bytree=colsample_bytree
    ).fit(X_processed.loc[t_idx, feature_cols], y.loc[t_idx])
    pred = model.predict(X_processed.loc[v_idx, feature_cols])
    y_v = y.loc[v_idx]
    return {
        "r2_v": r2_score(y_v, pred),
        "rmse_v": np.sqrt(mean_squared_error(y_v, pred)),
        "mae_v": mean_absolute_error(y_v, pred),
        "mape_v": mean_absolute_percentage_error(y_v, pred) * 100,
    }

# 2. Pengujian Ablasi Baseline pada Skenario Terbaik (SW-8)
cols_with = list(X_processed.columns)
cols_without = [col for col in X_processed.columns if col != "BULAN_BASAH"]

metrics_with = evaluate_sliding_window(cols_with, window_size=w)
metrics_without = evaluate_sliding_window(cols_without, window_size=w)

df_ablasi_produktivitas = pd.DataFrame({
    "Metrik Evaluasi": ["R²", "RMSE (Ton/Ha)", "MAE (Ton/Ha)", "MAPE (%)"],
    "Dengan Bulan Basah (Baseline)": [f"{metrics_with['r2_v']:.4f}", f"{metrics_with['rmse_v']:.4f}", f"{metrics_with['mae_v']:.4f}", f"{metrics_with['mape_v']:.2f}%"],
    "Tanpa Bulan Basah": [f"{metrics_without['r2_v']:.4f}", f"{metrics_without['rmse_v']:.4f}", f"{metrics_without['mae_v']:.4f}", f"{metrics_without['mape_v']:.2f}%"],
    "Selisih Performa (Δ)": [f"{metrics_without['r2_v'] - metrics_with['r2_v']:+.4f}", f"{metrics_without['rmse_v'] - metrics_with['rmse_v']:+.4f}", f"{metrics_without['mae_v'] - metrics_with['mae_v']:+.4f}", f"{metrics_without['mape_v'] - metrics_with['mape_v']:+.2f}%"],
})

df_ablasi_produksi = pd.DataFrame({
    "Metrik Evaluasi": ["R²", "RMSE (Ton)", "MAE (Ton)", "MAPE (%)"],
    "Dengan Bulan Basah (Baseline)": [f"{metrics_with['r2_p']:.4f}", f"{metrics_with['rmse_p']:,.2f}", f"{metrics_with['mae_p']:,.2f}", f"{metrics_with['mape_p']:.2f}%"],
    "Tanpa Bulan Basah": [f"{metrics_without['r2_p']:.4f}", f"{metrics_without['rmse_p']:,.2f}", f"{metrics_without['mae_p']:,.2f}", f"{metrics_without['mape_p']:.2f}%"],
    "Selisih Performa (Δ)": [f"{metrics_without['r2_p'] - metrics_with['r2_p']:+.4f}", f"{metrics_without['rmse_p'] - metrics_with['rmse_p']:+,.2f}", f"{metrics_without['mae_p'] - metrics_with['mae_p']:+,.2f}", f"{metrics_without['mape_p'] - metrics_with['mape_p']:+.2f}%"],
})

print(f"\nSkenario Uji: True Sliding Window ({w} Thn - Rata-rata Walk-Forward)")
print("\n" + "=" * 65)
print("TABEL 1. KOMPARASI ABLASI BULAN BASAH PADA PRODUKTIVITAS (TON/HA)")
print("=" * 65)
display(df_ablasi_produktivitas)

print("\n" + "=" * 65)
print("TABEL 2. KOMPARASI ABLASI BULAN BASAH PADA PRODUKSI (TON)")
print("=" * 65)
display(df_ablasi_produksi)

# 3. Replikasi Multi-Seed (10 Seeds) dengan Pelaporan Mean ± SD
#    subsample/colsample < 1 agar seed benar-benar memengaruhi model
print("\n" + "=" * 70)
print(f"TABEL 3. REPLIKASI ABLASI MULTI-SEED ({len(SEEDS)} SEEDS) PADA SW-{w} DENGAN MEAN ± SD")
print("=" * 70)

ms_with_r2, ms_without_r2 = [], []
ms_with_rmse, ms_without_rmse = [], []
ms_with_mape, ms_without_mape = [], []

for s in SEEDS:
    mw = evaluate_sliding_window(cols_with, window_size=w, random_state=s, subsample=0.8, colsample_bytree=0.8)
    mwo = evaluate_sliding_window(cols_without, window_size=w, random_state=s, subsample=0.8, colsample_bytree=0.8)
    ms_with_r2.append(mw['r2_v'])
    ms_without_r2.append(mwo['r2_v'])
    ms_with_rmse.append(mw['rmse_v'])
    ms_without_rmse.append(mwo['rmse_v'])
    ms_with_mape.append(mw['mape_v'])
    ms_without_mape.append(mwo['mape_v'])

t_stat, p_val = stats.ttest_rel(ms_with_r2, ms_without_r2)
n_seed_better = int(np.sum(np.array(ms_with_r2) > np.array(ms_without_r2)))

df_multiseed = pd.DataFrame({
    "Metrik (Produktivitas)": ["R²", "RMSE (Ton/Ha)", "MAPE (%)"],
    "Dengan Bulan Basah (Mean ± SD)": [
        f"{np.mean(ms_with_r2):.4f} ± {np.std(ms_with_r2, ddof=1):.4f}",
        f"{np.mean(ms_with_rmse):.4f} ± {np.std(ms_with_rmse, ddof=1):.4f}",
        f"{np.mean(ms_with_mape):.2f}% ± {np.std(ms_with_mape, ddof=1):.2f}%"
    ],
    "Tanpa Bulan Basah (Mean ± SD)": [
        f"{np.mean(ms_without_r2):.4f} ± {np.std(ms_without_r2, ddof=1):.4f}",
        f"{np.mean(ms_without_rmse):.4f} ± {np.std(ms_without_rmse, ddof=1):.4f}",
        f"{np.mean(ms_without_mape):.2f}% ± {np.std(ms_without_mape, ddof=1):.2f}%"
    ],
    "Selisih Bersih (Δ)": [
        f"{np.mean(ms_with_r2) - np.mean(ms_without_r2):+.4f}",
        f"{np.mean(ms_without_rmse) - np.mean(ms_with_rmse):+.4f}",
        f"{np.mean(ms_without_mape) - np.mean(ms_with_mape):+.2f}%"
    ]
})
display(df_multiseed)

sig_label = "SIGNIFIKAN (p < 0.05)" if p_val < 0.05 else "TIDAK SIGNIFIKAN (p ≥ 0.05)"
print(f"\nUji Signifikansi Statistik (Paired t-test R², n = {len(SEEDS)} seed): "
      f"t = {t_stat:.4f}, p-value = {p_val:.4f} -> {sig_label}")
print(f"Jumlah seed di mana model DENGAN BULAN_BASAH lebih baik: {n_seed_better} dari {len(SEEDS)}")
print("Catatan: variasi antar-seed hanya mencerminkan keacakan algoritma, bukan ketidakpastian sampel;")
print(f"jumlah tahun uji tetap {len(years_available) - w} tahun.")

# 4. Ablasi Multi-Seed Lintas Seluruh Skenario Temporal (7 Skenario)
print("\n" + "=" * 70)
print(f"TABEL 4. ABLASI MULTI-SEED ({len(SEEDS)} SEEDS) LINTAS SELURUH SKENARIO")
print("=" * 70)

scenario_defs = [(f"SW-{k} ({k} Thn)", 'sw', k) for k in [5, 6, 7, 8]] + [
    ('EW-1 (≤2020)', 'ew', 2020),
    ('EW-2 (≤2021)', 'ew', 2021),
    ('EW-3 (≤2022)', 'ew', 2022),
]

scen_rows = []
for name, kind, param in scenario_defs:
    d_r2, d_mape, r2_w_list, r2_wo_list = [], [], [], []
    for s in SEEDS:
        if kind == 'sw':
            res_w = evaluate_sliding_window(cols_with, window_size=param, random_state=s, subsample=0.8, colsample_bytree=0.8)
            res_wo = evaluate_sliding_window(cols_without, window_size=param, random_state=s, subsample=0.8, colsample_bytree=0.8)
        else:
            res_w = evaluate_expanding(cols_with, param, random_state=s, subsample=0.8, colsample_bytree=0.8)
            res_wo = evaluate_expanding(cols_without, param, random_state=s, subsample=0.8, colsample_bytree=0.8)
        r2_w_list.append(res_w['r2_v'])
        r2_wo_list.append(res_wo['r2_v'])
        d_r2.append(res_w['r2_v'] - res_wo['r2_v'])
        d_mape.append(res_wo['mape_v'] - res_w['mape_v'])
    t_s, p_s = stats.ttest_rel(r2_w_list, r2_wo_list)
    scen_rows.append({
        "Skenario": name,
        "R² Dengan (Mean ± SD)": f"{np.mean(r2_w_list):.4f} ± {np.std(r2_w_list, ddof=1):.4f}",
        "R² Tanpa (Mean ± SD)": f"{np.mean(r2_wo_list):.4f} ± {np.std(r2_wo_list, ddof=1):.4f}",
        "Δ R² (Mean)": round(np.mean(d_r2), 4),
        "Δ MAPE (%) (Mean)": round(np.mean(d_mape), 2),
        "Seed Δ R² > 0": f"{int(np.sum(np.array(d_r2) > 0))}/{len(SEEDS)}",
        "p-value": round(p_s, 4),
    })

df_all_scen = pd.DataFrame(scen_rows)
display(df_all_scen)

# 5. Permutation Feature Importance (OUT-OF-SAMPLE: dihitung pada tahun uji tiap fold)
print("\n" + "=" * 70)
print(f"TABEL 5. PERMUTATION IMPORTANCE OUT-OF-SAMPLE (SW-{w}, n_repeats=30)")
print("=" * 70)

perm_feats = X_COLS + ['YEAR_INDEX']
perm_acc = {f: [] for f in perm_feats}
for start_idx in range(len(years_available) - w):
    train_years = years_available[start_idx : start_idx + w]
    test_year = years_available[start_idx + w]
    t_idx = df_master[df_master['YEAR_fitur'].isin(train_years)].index
    v_idx = df_master[df_master['YEAR_fitur'] == test_year].index
    m_fold = xgb.XGBRegressor(random_state=42).fit(X_processed.loc[t_idx], y.loc[t_idx])
    perm = permutation_importance(
        m_fold, X_processed.loc[v_idx], y.loc[v_idx],
        scoring='neg_root_mean_squared_error', n_repeats=30, random_state=42
    )
    for feat in perm_feats:
        col_idx = X_processed.columns.get_loc(feat)
        perm_acc[feat].append(perm.importances_mean[col_idx])

df_perm_agro = pd.DataFrame({
    'Fitur Agroklimat & Temporal': perm_feats,
    'Kenaikan RMSE saat diacak (Ton/Ha, rata2 fold)': [round(np.mean(perm_acc[f]), 5) for f in perm_feats],
}).sort_values(by='Kenaikan RMSE saat diacak (Ton/Ha, rata2 fold)', ascending=False).reset_index(drop=True)
display(df_perm_agro)
print("Catatan: karena fitur iklim (hampir) sama untuk seluruh kecamatan dalam satu tahun uji,")
print("mengacak fitur tersebut di dalam satu tahun hampir tidak mengubah apa pun. Nilai mendekati nol")
print("berarti fitur tidak membedakan antar-kecamatan, BUKAN bahwa fitur tidak berpengaruh antar-tahun.")

# 6. Kesimpulan (seluruh angka diambil dari hasil di atas, tidak ada angka yang diketik manual)
bb_rank = int(df_perm_agro.index[df_perm_agro['Fitur Agroklimat & Temporal'] == 'BULAN_BASAH'][0]) + 1
n_scen_pos = int((df_all_scen['Δ R² (Mean)'] > 0).sum())
n_scen_sig = int((df_all_scen['p-value'] < 0.05).sum())

print("\nKesimpulan Analisis Ablasi & Feature Importance (otomatis):")
print(f"1. Single-run SW-{w} (seed 42): R² {metrics_with['r2_v']:.4f} (dengan) vs {metrics_without['r2_v']:.4f} (tanpa), "
      f"Δ MAPE = {metrics_without['mape_v'] - metrics_with['mape_v']:+.2f} poin persen.")
print(f"2. Multi-seed SW-{w}: Δ R² rata-rata = {np.mean(ms_with_r2) - np.mean(ms_without_r2):+.4f}, "
      f"p = {p_val:.4f} -> {sig_label}.")
print(f"3. Lintas skenario: Δ R² rata-rata positif di {n_scen_pos} dari {len(df_all_scen)} skenario; "
      f"signifikan (p < 0.05) di {n_scen_sig} skenario.")
print(f"4. Permutation importance out-of-sample: BULAN_BASAH berada di peringkat {bb_rank} dari {len(perm_feats)}.")


=== PENGUJIAN KOMPARASI FITUR 'BULAN_BASAH' (ABLATION TEST) ===



Skenario Uji: True Sliding Window (7 Thn - Rata-rata Walk-Forward)

TABEL 1. KOMPARASI ABLASI BULAN BASAH PADA PRODUKTIVITAS (TON/HA)


,Metrik Evaluasi,Dengan Bulan Basah (Baseline),Tanpa Bulan Basah,Selisih Performa (Δ)
0,R²,0.3594,0.4273,+0.0678
1,RMSE (Ton/Ha),0.3305,0.3122,-0.0183
2,MAE (Ton/Ha),0.2372,0.2195,-0.0177
3,MAPE (%),4.07%,3.79%,-0.29%



TABEL 2. KOMPARASI ABLASI BULAN BASAH PADA PRODUKSI (TON)


,Metrik Evaluasi,Dengan Bulan Basah (Baseline),Tanpa Bulan Basah,Selisih Performa (Δ)
0,R²,0.9840,0.9863,+0.0023
1,RMSE (Ton),883.26,816.75,-66.51
2,MAE (Ton),572.30,519.81,-52.49
3,MAPE (%),4.07%,3.79%,-0.29%



TABEL 3. REPLIKASI ABLASI MULTI-SEED (10 SEEDS) PADA SW-7 DENGAN MEAN ± SD


... (2 outputs left out)

,Skenario,R² Dengan (Mean ± SD),R² Tanpa (Mean ± SD),Δ R² (Mean),Δ MAPE (%) (Mean),Seed Δ R² > 0,p-value
0,SW-5 (5 Thn),-0.0768 ± 0.2430,-0.1846 ± 0.2311,0.1078,0.16,8/10,0.3559
1,SW-6 (6 Thn),-0.1728 ± 0.1696,-0.4406 ± 0.3204,0.2678,0.50,6/10,0.0598
2,SW-7 (7 Thn),0.2528 ± 0.1021,0.3016 ± 0.0753,-0.0489,-0.13,3/10,0.2115
3,SW-8 (8 Thn),0.2334 ± 0.0910,0.2643 ± 0.1281,-0.0309,-0.11,3/10,0.5099
4,EW-1 (≤2020),-0.0754 ± 0.1717,0.0211 ± 0.0975,-0.0965,-0.19,3/10,0.1599
5,EW-2 (≤2021),0.2349 ± 0.1073,0.2526 ± 0.1547,-0.0177,-0.10,5/10,0.7691
6,EW-3 (≤2022),0.3144 ± 0.0705,0.3492 ± 0.0772,-0.0348,-0.01,5/10,0.2926



TABEL 5. PERMUTATION IMPORTANCE OUT-OF-SAMPLE (SW-7, n_repeats=30)


,Fitur Agroklimat & Temporal,"Kenaikan RMSE saat diacak (Ton/Ha, rata2 fold)"
0,TOTAL_HUJAN,0.01112
1,MIN_SUHU_THN,0.00083
2,AVG_RADIASI,0.00021
3,BULAN_BASAH,0.00009
4,AVG_KELEMBABAN,0.00008
5,MAX_SUHU_THN,0.00000
6,AVG_SUHU,0.00000
7,YEAR_INDEX,0.00000


Catatan: karena fitur iklim (hampir) sama untuk seluruh kecamatan dalam satu tahun uji,
mengacak fitur tersebut di dalam satu tahun hampir tidak mengubah apa pun. Nilai mendekati nol
berarti fitur tidak membedakan antar-kecamatan, BUKAN bahwa fitur tidak berpengaruh antar-tahun.

Kesimpulan Analisis Ablasi & Feature Importance (otomatis):
1. Single-run SW-7 (seed 42): R² 0.3594 (dengan) vs 0.4273 (tanpa), Δ MAPE = -0.29 poin persen.
2. Multi-seed SW-7: Δ R² rata-rata = -0.0489, p = 0.2115 -> TIDAK SIGNIFIKAN (p ≥ 0.05).
3. Lintas skenario: Δ R² rata-rata positif di 2 dari 7 skenario; signifikan (p < 0.05) di 0 skenario.
4. Permutation importance out-of-sample: BULAN_BASAH berada di peringkat 4 dari 8.


## EKSPERIMEN 1: HYPERPARAMETER TUNING XGBOOST DENGAN OPTUNA (NESTED)
Optimasi Bayesian (Optuna TPE) dengan skema *nested walk-forward*: untuk setiap fold uji, hyperparameter dipilih hanya dari data di dalam jendela latih (validasi internal pada 2 tahun terakhir jendela). Tahun uji tidak pernah dipakai untuk memilih hyperparameter, sehingga metrik yang dilaporkan bebas bias seleksi.

In [4]:
# =========================================================
# EKSPERIMEN 1: HYPERPARAMETER TUNING DENGAN OPTUNA (NESTED, BEBAS KEBOCORAN)
# Optuna HANYA melihat tahun-tahun di dalam jendela latih. Validasi internal
# memakai walk-forward pada 2 tahun terakhir jendela latih; tahun uji luar
# (outer test) tidak pernah dipakai untuk memilih hyperparameter.
# =========================================================

import optuna
import pandas as pd
import numpy as np
import xgboost as xgb
from IPython.display import display
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error
)

optuna.logging.set_verbosity(optuna.logging.WARNING)

print("=== OPTIMASI HYPERPARAMETER XGBOOST (NESTED WALK-FORWARD) ===")

w = 8                # Window length skenario utama
INNER_VAL_YEARS = 2  # Jumlah tahun validasi internal di akhir jendela latih
N_TRIALS = 40

def suggest_params(trial):
    return {
        'max_depth': trial.suggest_int('max_depth', 2, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 50, 400, step=50),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 10.0, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 1.0, log=True),
        'random_state': 42
    }

def tune_on_window(train_years, n_trials=N_TRIALS, seed=42):
    """Cari hyperparameter memakai tahun-tahun di dalam train_years saja."""
    train_years = list(train_years)
    inner_w = len(train_years) - INNER_VAL_YEARS

    def objective(trial):
        params = suggest_params(trial)
        sq_err = []
        for k in range(INNER_VAL_YEARS):
            tr = train_years[k : k + inner_w]
            va = train_years[k + inner_w]
            ti = df_master[df_master['YEAR_fitur'].isin(tr)].index
            vi = df_master[df_master['YEAR_fitur'] == va].index
            m = xgb.XGBRegressor(**params).fit(X_processed.loc[ti], y.loc[ti])
            sq_err.append(mean_squared_error(y.loc[vi], m.predict(X_processed.loc[vi])))
        return np.sqrt(np.mean(sq_err))

    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=seed))
    study.optimize(objective, n_trials=n_trials)
    best = study.best_params.copy()
    best['random_state'] = 42
    return best, study.best_value

def nested_walk_forward(window, test_feature_years=None, n_trials=N_TRIALS):
    """Outer loop walk-forward. Setiap fold di-tuning ulang hanya dengan data latihnya."""
    if test_feature_years is None:
        test_feature_years = years_available[window:]
    pred_rows, fold_rows = [], []
    for tf in test_feature_years:
        tr_years = [yr for yr in years_available if tf - window <= yr < tf]
        best, inner_rmse = tune_on_window(tr_years, n_trials)
        t_idx = df_master[df_master['YEAR_fitur'].isin(tr_years)].index
        v_idx = df_master[df_master['YEAR_fitur'] == tf].index

        pred_def = xgb.XGBRegressor(random_state=42).fit(X_processed.loc[t_idx], y.loc[t_idx]).predict(X_processed.loc[v_idx])
        pred_tun = xgb.XGBRegressor(**best).fit(X_processed.loc[t_idx], y.loc[t_idx]).predict(X_processed.loc[v_idx])

        pred_rows.append(pd.DataFrame({
            'WADMKC': df_master.loc[v_idx, 'WADMKC'].values,
            'YEAR_fitur': tf,
            'YEAR_TARGET': tf + 1,
            'LUAS_PANEN': df_master.loc[v_idx, 'LUAS_PANEN'].values,
            'y_true': y.loc[v_idx].values,
            'pred_default': pred_def,
            'pred_tuned': pred_tun,
        }, index=v_idx))
        fold_rows.append({'Tahun Target Uji': tf + 1, 'Inner RMSE (validasi internal)': round(inner_rmse, 4),
                          **{k: (round(v, 4) if isinstance(v, float) else v) for k, v in best.items() if k != 'random_state'}})
    return pd.DataFrame(fold_rows), pd.concat(pred_rows)

def summarize_predictions(df_pred, pred_col):
    """Metrik rata-rata per fold (seperti Tabel naskah) + pooled."""
    per_fold = []
    for _, g in df_pred.groupby('YEAR_TARGET'):
        per_fold.append({
            'r2': r2_score(g['y_true'], g[pred_col]),
            'rmse': np.sqrt(mean_squared_error(g['y_true'], g[pred_col])),
            'mae': mean_absolute_error(g['y_true'], g[pred_col]),
            'mape': mean_absolute_percentage_error(g['y_true'], g[pred_col]) * 100,
        })
    per_fold = pd.DataFrame(per_fold)
    return {
        'R² (Mean Fold)': round(per_fold['r2'].mean(), 4),
        'R² (Pooled)': round(r2_score(df_pred['y_true'], df_pred[pred_col]), 4),
        'RMSE (Ton/Ha, Pooled)': round(np.sqrt(mean_squared_error(df_pred['y_true'], df_pred[pred_col])), 4),
        'MAE (Ton/Ha)': round(mean_absolute_error(df_pred['y_true'], df_pred[pred_col]), 4),
        'MAPE (%)': round(mean_absolute_percentage_error(df_pred['y_true'], df_pred[pred_col]) * 100, 2),
    }

print(f"Skenario: True Sliding Window ({w} Thn) | Validasi internal: {INNER_VAL_YEARS} tahun terakhir jendela latih")
print(f"Menjalankan nested Optuna TPE ({N_TRIALS} trial per fold)...")
df_folds_sw8, df_oos_sw8 = nested_walk_forward(w)

print("\n" + "=" * 70)
print("TABEL 1. HYPERPARAMETER TERPILIH PER FOLD (DIPILIH TANPA MELIHAT DATA UJI)")
print("=" * 70)
display(df_folds_sw8)

df_res_v = pd.DataFrame([
    {'Model': 'XGBoost Default (Baseline)', **summarize_predictions(df_oos_sw8, 'pred_default')},
    {'Model': 'XGBoost Tuned (Nested Optuna)', **summarize_predictions(df_oos_sw8, 'pred_tuned')},
])

df_oos_sw8['prod_true'] = df_oos_sw8['y_true'] * df_oos_sw8['LUAS_PANEN']
df_res_p = pd.DataFrame([
    {'Model': name,
     'R² (Pooled)': round(r2_score(df_oos_sw8['prod_true'], df_oos_sw8[col] * df_oos_sw8['LUAS_PANEN']), 4),
     'RMSE (Ton)': round(np.sqrt(mean_squared_error(df_oos_sw8['prod_true'], df_oos_sw8[col] * df_oos_sw8['LUAS_PANEN'])), 2),
     'MAE (Ton)': round(mean_absolute_error(df_oos_sw8['prod_true'], df_oos_sw8[col] * df_oos_sw8['LUAS_PANEN']), 2)}
    for name, col in [('XGBoost Default (Baseline)', 'pred_default'), ('XGBoost Tuned (Nested Optuna)', 'pred_tuned')]
])

print("\n" + "=" * 70)
print("TABEL 2. DEFAULT VS NESTED OPTUNA PADA PRODUKTIVITAS (TON/HA)")
print("=" * 70)
display(df_res_v)

print("\n" + "=" * 70)
print("TABEL 3. DEFAULT VS NESTED OPTUNA PADA PRODUKSI (TON) - HASIL KONVERSI")
print("=" * 70)
display(df_res_p)

# Hyperparameter final untuk proyeksi 2025: di-tuning pada jendela latih terakhir saja
FINAL_TRAIN_YEARS = list(years_available[-w:])
FINAL_PARAMS, final_inner_rmse = tune_on_window(FINAL_TRAIN_YEARS)
RMSE_OOS_TUNED = float(df_res_v.loc[1, 'RMSE (Ton/Ha, Pooled)'])
best_params = FINAL_PARAMS.copy()

delta_r2 = df_res_v.loc[1, 'R² (Mean Fold)'] - df_res_v.loc[0, 'R² (Mean Fold)']
delta_rmse = df_res_v.loc[1, 'RMSE (Ton/Ha, Pooled)'] - df_res_v.loc[0, 'RMSE (Ton/Ha, Pooled)']

print("\nRingkasan (otomatis):")
print(f"-> Δ R² (Mean Fold) tuned - default : {delta_r2:+.4f} ({df_res_v.loc[0, 'R² (Mean Fold)']} -> {df_res_v.loc[1, 'R² (Mean Fold)']})")
print(f"-> Δ RMSE pooled tuned - default    : {delta_rmse:+.4f} ton/ha")
print(f"-> Hyperparameter final (jendela {min(FINAL_TRAIN_YEARS)}-{max(FINAL_TRAIN_YEARS)}): {FINAL_PARAMS}")
print("Catatan: angka R² = 0.7056 pada versi sebelumnya diperoleh dengan men-tuning langsung pada tahun uji")
print("sehingga bias optimistis; angka di atas adalah estimasi yang bebas kebocoran.")


C:\Users\acer\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== OPTIMASI HYPERPARAMETER XGBOOST (NESTED WALK-FORWARD) ===
Skenario: True Sliding Window (8 Thn) | Validasi internal: 2 tahun terakhir jendela latih
Menjalankan nested Optuna TPE (40 trial per fold)...



TABEL 1. HYPERPARAMETER TERPILIH PER FOLD (DIPILIH TANPA MELIHAT DATA UJI)


,Tahun Target Uji,Inner RMSE (validasi internal),max_depth,learning_rate,n_estimators,min_child_weight,subsample,colsample_bytree,reg_lambda,reg_alpha
0,2023,0.2994,3,0.0900,100,1,0.8983,0.5220,0.3965,0.0274
1,2024,0.3613,2,0.1103,100,2,0.8412,0.5196,0.2058,0.1346



TABEL 2. DEFAULT VS NESTED OPTUNA PADA PRODUKTIVITAS (TON/HA)


,Model,R² (Mean Fold),R² (Pooled),"RMSE (Ton/Ha, Pooled)",MAE (Ton/Ha),MAPE (%)
0,XGBoost Default (Baseline),0.2424,0.2421,0.3577,0.2620,4.46
1,XGBoost Tuned (Nested Optuna),0.3225,0.3221,0.3383,0.2531,4.33



TABEL 3. DEFAULT VS NESTED OPTUNA PADA PRODUKSI (TON) - HASIL KONVERSI


,Model,R² (Pooled),RMSE (Ton),MAE (Ton)
0,XGBoost Default (Baseline),0.9791,1029.36,651.23
1,XGBoost Tuned (Nested Optuna),0.9843,892.43,567.03



Ringkasan (otomatis):
-> Δ R² (Mean Fold) tuned - default : +0.0801 (0.2424 -> 0.3225)
-> Δ RMSE pooled tuned - default    : -0.0194 ton/ha
-> Hyperparameter final (jendela 2016-2023): {'max_depth': 4, 'learning_rate': 0.13409646757763521, 'n_estimators': 250, 'min_child_weight': 5, 'subsample': 0.9280344893809704, 'colsample_bytree': 0.5374578244625685, 'reg_lambda': 0.21735368731968732, 'reg_alpha': 0.0030116236366303484, 'random_state': 42}
Catatan: angka R² = 0.7056 pada versi sebelumnya diperoleh dengan men-tuning langsung pada tahun uji
sehingga bias optimistis; angka di atas adalah estimasi yang bebas kebocoran.


## EKSPERIMEN 2: KOMPARASI KRITERIA BULAN BASAH OLDEMAN (KONTINU VS TOTAL)
Eksperimen ini menjawab langsung masukan Reviewer 1 terkait kriteria agroklimat Oldeman yang mendasarkan klasifikasi musim pada deret bulan basah berurutan (*consecutive wet months*, >200 mm/bulan) dibandingkan dengan total hitungan bulan basah kalender tahunan.

In [5]:
# =========================================================
# EKSPERIMEN 2: BULAN BASAH BERURUTAN (OLDEMAN) VS TOTAL BB
# Dua varian deret berurutan:
#   (a) Kalender      : Jan-Des tahun t (deret musim hujan terpotong di pergantian tahun)
#   (b) Tahun musim   : Jul(t-1) - Jun(t), sehingga musim hujan Okt-Apr tidak terpotong
#                       dan tetap hanya memakai informasi s.d. tahun t (tanpa kebocoran)
# Seluruh varian dievaluasi dengan XGBoost default agar perbandingan adil
# (hyperparameter tidak dipilih dari data uji).
# =========================================================

import geopandas as gpd
import numpy as np
import pandas as pd
import xgboost as xgb
from IPython.display import display
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error
)

print("=== EVALUASI KRITERIA BULAN BASAH: BERURUTAN (OLDEMAN) VS TOTAL TAHUNAN ===")

WET_MM = 200  # ambang bulan basah Oldeman (mm/bulan)
month_cols = ['JAN', 'FEB', 'MAR', 'APR', 'MAY', 'JUN', 'JUL', 'AUG', 'SEP', 'OCT', 'NOV', 'DEC']

# ---------------------------------------------------------
# 1. Data bulanan MERRA-2 & pemetaan grid kecamatan
# ---------------------------------------------------------
df_ch = pd.read_csv("Curah-hujan.csv")
pts_merra = df_ch[['LAT', 'LON']].drop_duplicates()
gdf_merra = gpd.GeoDataFrame(
    pts_merra,
    geometry=gpd.points_from_xy(pts_merra['LON'], pts_merra['LAT']),
    crs="EPSG:4326"
).to_crs(epsg=32749)

joined_ch = gpd.sjoin_nearest(gdf_banyumas_utm, gdf_merra, how="inner")
subdistrict_ch_grid = joined_ch[['WADMKC', 'LAT', 'LON']].drop_duplicates(subset=['WADMKC']).copy()

def max_run(values):
    max_c, cur_c = 0, 0
    for v in values:
        cur_c = cur_c + 1 if v > WET_MM else 0
        max_c = max(max_c, cur_c)
    return max_c

df_ch = df_ch.sort_values(['LAT', 'LON', 'YEAR']).reset_index(drop=True)
df_ch['BB_TOTAL_CEK'] = (df_ch[month_cols] > WET_MM).sum(axis=1)
df_ch['BULAN_BASAH_KONTINU'] = df_ch[month_cols].apply(max_run, axis=1)

prev = df_ch.groupby(['LAT', 'LON'])[month_cols].shift(1)
season_runs = []
for i in range(len(df_ch)):
    if prev.loc[i].isna().any():
        season_runs.append(np.nan)  # tahun pertama (2014) tidak punya data Jul-Des tahun sebelumnya
    else:
        seq = list(prev.loc[i, month_cols[6:]]) + list(df_ch.loc[i, month_cols[:6]])
        season_runs.append(max_run(seq))
df_ch['BB_KONTINU_MUSIM'] = season_runs

df_consec = pd.merge(
    subdistrict_ch_grid,
    df_ch[['YEAR', 'LAT', 'LON', 'BB_TOTAL_CEK', 'BULAN_BASAH_KONTINU', 'BB_KONTINU_MUSIM']],
    on=['LAT', 'LON'],
    how='inner'
)
df_consec['WADMKC'] = (
    df_consec['WADMKC'].str.title().str.strip()
    .replace({'Baturraden': 'Baturaden', 'Tambak': 'Tampak'})
)

# ---------------------------------------------------------
# 2. Integrasi ke dataset lagging (fitur t -> target t+1)
# ---------------------------------------------------------
df_eng = pd.merge(
    df_dataset,
    df_consec[['WADMKC', 'YEAR', 'BB_TOTAL_CEK', 'BULAN_BASAH_KONTINU', 'BB_KONTINU_MUSIM']],
    on=['WADMKC', 'YEAR'],
    how='left'
)
cek_match = (df_eng['BB_TOTAL_CEK'] == df_eng['BULAN_BASAH']).mean() * 100
print(f"Sanity check: total bulan basah (>{WET_MM} mm) hasil hitung ulang sama dengan kolom BULAN_BASAH "
      f"pada {cek_match:.1f}% baris.")

EXTRA_BB = ['BULAN_BASAH_KONTINU', 'BB_KONTINU_MUSIM']
df_feat_eng = df_eng[['WADMKC', 'YEAR'] + X_COLS + EXTRA_BB].copy()
df_feat_eng['YEAR_TARGET'] = df_feat_eng['YEAR'] + 1

df_m_eng = pd.merge(
    df_feat_eng,
    df_target,
    left_on=['WADMKC', 'YEAR_TARGET'],
    right_on=['WADMKC', 'YEAR'],
    how='inner',
    suffixes=('_fitur', '_target')
)

years_eng = sorted(df_m_eng['YEAR_fitur'].unique())
w = 8

# ---------------------------------------------------------
# 3. Komparasi indikator bulan basah
# ---------------------------------------------------------
base_cols = [c for c in X_COLS if c != 'BULAN_BASAH']
bb_scenarios = [
    {'name': '0. Tanpa Indikator Bulan Basah', 'cols': base_cols},
    {'name': '1. Total Bulan Basah Tahunan (Fitur Naskah)', 'cols': X_COLS},
    {'name': '2. BB Berurutan - Kalender Jan-Des', 'cols': base_cols + ['BULAN_BASAH_KONTINU']},
    {'name': '3. BB Berurutan - Tahun Musim Jul-Jun', 'cols': base_cols + ['BB_KONTINU_MUSIM']},
    {'name': '4. Total BB + BB Berurutan Tahun Musim', 'cols': X_COLS + ['BB_KONTINU_MUSIM']},
]

results_bb_v = []
for sc in bb_scenarios:
    X_raw_sc = df_m_eng[sc['cols'] + ['WADMKC', 'YEAR_fitur']]
    X_proc_sc = pd.get_dummies(X_raw_sc, columns=['WADMKC'], dtype=int)
    X_proc_sc = X_proc_sc.rename(columns={'YEAR_fitur': 'YEAR_INDEX'})
    y_sc = df_m_eng['PRODUKTIVITAS']

    r2_v, yt_all, yp_all = [], [], []
    for s_idx in range(len(years_eng) - w):
        tr_y = years_eng[s_idx : s_idx + w]
        te_y = years_eng[s_idx + w]
        t_idx = df_m_eng[df_m_eng['YEAR_fitur'].isin(tr_y)].index
        v_idx = df_m_eng[df_m_eng['YEAR_fitur'] == te_y].index

        m = xgb.XGBRegressor(random_state=42).fit(X_proc_sc.loc[t_idx], y_sc.loc[t_idx])
        y_pred_prod = m.predict(X_proc_sc.loc[v_idx])
        r2_v.append(r2_score(y_sc.loc[v_idx], y_pred_prod))
        yt_all.extend(y_sc.loc[v_idx]); yp_all.extend(y_pred_prod)

    results_bb_v.append({
        'Indikator Bulan Basah': sc['name'],
        'R² (Mean Fold)': round(np.mean(r2_v), 4),
        'RMSE (Ton/Ha, Pooled)': round(np.sqrt(mean_squared_error(yt_all, yp_all)), 4),
        'MAE (Ton/Ha)': round(mean_absolute_error(yt_all, yp_all), 4),
        'MAPE (%)': round(mean_absolute_percentage_error(yt_all, yp_all) * 100, 2)
    })

df_bb_v = pd.DataFrame(results_bb_v)

print("\n" + "=" * 70)
print(f"TABEL 1. KOMPARASI INDIKATOR BULAN BASAH (SW-{w}, XGBOOST DEFAULT, TON/HA)")
print("=" * 70)
display(df_bb_v)

corr_total_season = df_m_eng[['BULAN_BASAH', 'BB_KONTINU_MUSIM']].corr().iloc[0, 1]
r2_total = df_bb_v.loc[1, 'R² (Mean Fold)']
r2_season = df_bb_v.loc[3, 'R² (Mean Fold)']
r2_none = df_bb_v.loc[0, 'R² (Mean Fold)']

print("\nKesimpulan (otomatis):")
print(f"1. R² total bulan basah = {r2_total:.4f}; BB berurutan (tahun musim) = {r2_season:.4f}; "
      f"tanpa indikator = {r2_none:.4f}.")
print(f"2. Korelasi total BB vs BB berurutan (tahun musim) = {corr_total_season:.3f}.")
print("3. Bila total BB tetap dipakai, sebut fitur tersebut sebagai 'hitungan bulan basah dengan ambang")
print("   Oldeman (>200 mm)', bukan klasifikasi Oldeman, karena Oldeman didasarkan pada bulan basah berurutan.")


=== EVALUASI KRITERIA BULAN BASAH: BERURUTAN (OLDEMAN) VS TOTAL TAHUNAN ===


Sanity check: total bulan basah (>200 mm) hasil hitung ulang sama dengan kolom BULAN_BASAH pada 100.0% baris.



TABEL 1. KOMPARASI INDIKATOR BULAN BASAH (SW-8, XGBOOST DEFAULT, TON/HA)


,Indikator Bulan Basah,R² (Mean Fold),"RMSE (Ton/Ha, Pooled)",MAE (Ton/Ha),MAPE (%)
0,0. Tanpa Indikator Bulan Basah,-0.0050,0.4121,0.3010,5.09
1,1. Total Bulan Basah Tahunan (Fitur Naskah),0.2424,0.3577,0.2620,4.46
2,2. BB Berurutan - Kalender Jan-Des,0.1315,0.3831,0.2886,4.90
3,3. BB Berurutan - Tahun Musim Jul-Jun,0.1550,0.3779,0.2747,4.68
4,4. Total BB + BB Berurutan Tahun Musim,0.3477,0.3319,0.2298,3.94



Kesimpulan (otomatis):
1. R² total bulan basah = 0.2424; BB berurutan (tahun musim) = 0.1550; tanpa indikator = -0.0050.
2. Korelasi total BB vs BB berurutan (tahun musim) = 0.500.
3. Bila total BB tetap dipakai, sebut fitur tersebut sebagai 'hitungan bulan basah dengan ambang
   Oldeman (>200 mm)', bukan klasifikasi Oldeman, karena Oldeman didasarkan pada bulan basah berurutan.


## EKSPERIMEN 3: MODEL PEMBANDING (BASELINE BENCHMARKING)
Menjawab Reviewer 1 & 2: *naive persistence* (Y_t), persistence + drift, *subdistrict training-mean*, regresi linear, ridge, random forest, XGBoost default, XGBoost nested-tuned, serta model **anomali** (memprediksi perubahan produktivitas dari fitur iklim). Kolom *Skill vs Persistence* menunjukkan apakah model lebih baik dari persistence.

In [6]:
# =========================================================
# EKSPERIMEN 3: MODEL PEMBANDING (BASELINE BENCHMARKING)
# Evaluasi pada Produktivitas (Ton/Ha) - True Sliding Window (8 Thn)
# Termasuk persistence (Y_t), dan model ANOMALI: memprediksi perubahan
# produktivitas (Y_t+1 - Y_t) dari fitur iklim, lalu ditambahkan ke Y_t.
# =========================================================

import numpy as np
import pandas as pd
import xgboost as xgb
from IPython.display import display
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error
)

print("=== EVALUASI MODEL PEMBANDING (BASELINE BENCHMARKING) ===")

w = 8
ANOM_COLS = X_COLS + ['YEAR_fitur']
ANOM_COLS_NO_BB = [c for c in ANOM_COLS if c != 'BULAN_BASAH']
RIDGE_ALPHAS = [0.1, 1, 10, 100, 1000]
print(f"Skenario: True Sliding Window ({w} Thn) | Target Evaluasi: Produktivitas (Ton/Ha)")

def fit_ridge_anomaly(t_idx, cols, alpha):
    return make_pipeline(StandardScaler(), Ridge(alpha=alpha)).fit(
        df_master.loc[t_idx, cols], df_master.loc[t_idx, 'DELTA_PRODUKTIVITAS']
    )

def choose_alpha(train_years, cols):
    """Alpha dipilih dengan validasi internal (2 tahun terakhir jendela latih)."""
    train_years = list(train_years)
    inner_w = len(train_years) - INNER_VAL_YEARS
    scores = {}
    for a in RIDGE_ALPHAS:
        errs = []
        for k in range(INNER_VAL_YEARS):
            ti = df_master[df_master['YEAR_fitur'].isin(train_years[k : k + inner_w])].index
            vi = df_master[df_master['YEAR_fitur'] == train_years[k + inner_w]].index
            pred = df_master.loc[vi, 'PRODUKTIVITAS_T'] + fit_ridge_anomaly(ti, cols, a).predict(df_master.loc[vi, cols])
            errs.append(mean_squared_error(y.loc[vi], pred))
        scores[a] = np.mean(errs)
    return min(scores, key=scores.get)

def predict_model(m_name, tr_y, t_idx, v_idx):
    if m_name == 'Naive Persistence (Y_t)':
        return df_master.loc[v_idx, 'PRODUKTIVITAS_T'].values
    if m_name == 'Persistence + Drift (Y_t + rata2 Δ latih)':
        return df_master.loc[v_idx, 'PRODUKTIVITAS_T'].values + df_master.loc[t_idx, 'DELTA_PRODUKTIVITAS'].mean()
    if m_name == 'Subdistrict Training-Mean':
        tr_df = df_master.loc[t_idx]
        mean_map = tr_df.groupby('WADMKC')['PRODUKTIVITAS'].mean()
        return df_master.loc[v_idx, 'WADMKC'].map(mean_map).fillna(tr_df['PRODUKTIVITAS'].mean()).values
    if m_name.startswith('Anomali Ridge'):
        cols = ANOM_COLS_NO_BB if 'tanpa' in m_name else ANOM_COLS
        alpha = choose_alpha(tr_y, cols)
        return (df_master.loc[v_idx, 'PRODUKTIVITAS_T']
                + fit_ridge_anomaly(t_idx, cols, alpha).predict(df_master.loc[v_idx, cols])).values
    model_obj = feature_models[m_name]
    model_obj.fit(X_processed.loc[t_idx], y.loc[t_idx])
    return model_obj.predict(X_processed.loc[v_idx])

feature_models = {
    'Multiple Linear Regression': LinearRegression(),
    'Ridge Regression (alpha=1)': Ridge(alpha=1.0),
    'Random Forest (100 Trees)': RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42),
    'XGBoost Default': xgb.XGBRegressor(random_state=42),
}
model_order = [
    'Naive Persistence (Y_t)',
    'Persistence + Drift (Y_t + rata2 Δ latih)',
    'Subdistrict Training-Mean',
    'Multiple Linear Regression',
    'Ridge Regression (alpha=1)',
    'Random Forest (100 Trees)',
    'XGBoost Default',
    'Anomali Ridge (iklim, dengan BULAN_BASAH)',
    'Anomali Ridge (iklim, tanpa BULAN_BASAH)',
]

# ---------------------------------------------------------
# Kumpulkan prediksi out-of-sample seluruh model pada fold yang sama
# ---------------------------------------------------------
df_oos_all = df_oos_sw8[['WADMKC', 'YEAR_fitur', 'YEAR_TARGET', 'y_true']].copy()
for m_name in model_order:
    preds = pd.Series(index=df_oos_all.index, dtype=float)
    for tf in sorted(df_oos_all['YEAR_fitur'].unique()):
        tr_y = [yr for yr in years_available if tf - w <= yr < tf]
        t_idx = df_master[df_master['YEAR_fitur'].isin(tr_y)].index
        v_idx = df_master[df_master['YEAR_fitur'] == tf].index
        preds.loc[v_idx] = predict_model(m_name, tr_y, t_idx, v_idx)
    df_oos_all[m_name] = preds
df_oos_all['XGBoost Tuned (Nested Optuna)'] = df_oos_sw8['pred_tuned']
model_order.insert(7, 'XGBoost Tuned (Nested Optuna)')

mse_persist = mean_squared_error(df_oos_all['y_true'], df_oos_all['Naive Persistence (Y_t)'])
results_benchmark = []
for m_name in model_order:
    s = summarize_predictions(df_oos_all, m_name)
    s['Skill vs Persistence'] = round(1 - mean_squared_error(df_oos_all['y_true'], df_oos_all[m_name]) / mse_persist, 3)
    results_benchmark.append({'Model / Baseline': m_name, **s})

df_bench = pd.DataFrame(results_benchmark)

print("\n" + "=" * 70)
print("TABEL KOMPARASI MODEL PEMBANDING PADA TINGKAT PRODUKTIVITAS (TON/HA)")
print("=" * 70)
display(df_bench)
print("Skill vs Persistence = 1 - MSE_model / MSE_persistence (> 0 berarti lebih baik dari persistence).")

# ---------------------------------------------------------
# Kesimpulan otomatis
# ---------------------------------------------------------
best_row = df_bench.loc[df_bench['RMSE (Ton/Ha, Pooled)'].idxmin()]
feature_based = df_bench[~df_bench['Model / Baseline'].isin(
    ['Naive Persistence (Y_t)', 'Persistence + Drift (Y_t + rata2 Δ latih)', 'Subdistrict Training-Mean'])]
best_feat = feature_based.loc[feature_based['RMSE (Ton/Ha, Pooled)'].idxmin()]
row_xgb_t = df_bench[df_bench['Model / Baseline'] == 'XGBoost Tuned (Nested Optuna)'].iloc[0]
row_anom = df_bench[df_bench['Model / Baseline'] == 'Anomali Ridge (iklim, dengan BULAN_BASAH)'].iloc[0]
row_anom_nobb = df_bench[df_bench['Model / Baseline'] == 'Anomali Ridge (iklim, tanpa BULAN_BASAH)'].iloc[0]

print("\nKesimpulan (otomatis):")
print(f"1. Model dengan RMSE terendah: {best_row['Model / Baseline']} (RMSE = {best_row['RMSE (Ton/Ha, Pooled)']}, "
      f"MAPE = {best_row['MAPE (%)']}%).")
print(f"2. Model berbasis fitur terbaik: {best_feat['Model / Baseline']} "
      f"(skill vs persistence = {best_feat['Skill vs Persistence']:+.3f}).")
print(f"3. XGBoost Tuned (nested): RMSE = {row_xgb_t['RMSE (Ton/Ha, Pooled)']}, "
      f"skill vs persistence = {row_xgb_t['Skill vs Persistence']:+.3f}.")
print(f"4. Model anomali iklim: dengan BULAN_BASAH skill = {row_anom['Skill vs Persistence']:+.3f}, "
      f"tanpa BULAN_BASAH skill = {row_anom_nobb['Skill vs Persistence']:+.3f}.")
if best_feat['Skill vs Persistence'] <= 0:
    print("-> Tidak ada model berbasis fitur yang mengalahkan persistence. Klaim keunggulan model harus")
    print("   dinyatakan relatif terhadap persistence, dan kontribusi iklim dibahas sebagai keterbatasan.")


=== EVALUASI MODEL PEMBANDING (BASELINE BENCHMARKING) ===
Skenario: True Sliding Window (8 Thn) | Target Evaluasi: Produktivitas (Ton/Ha)



TABEL KOMPARASI MODEL PEMBANDING PADA TINGKAT PRODUKTIVITAS (TON/HA)


,Model / Baseline,R² (Mean Fold),R² (Pooled),"RMSE (Ton/Ha, Pooled)",MAE (Ton/Ha),MAPE (%),Skill vs Persistence
0,Naive Persistence (Y_t),0.9707,0.9706,0.0704,0.0495,0.85,0.000
1,Persistence + Drift (Y_t + rata2 Δ latih),0.9711,0.9711,0.0699,0.0606,1.04,0.016
2,Subdistrict Training-Mean,0.5505,0.5504,0.2755,0.2046,3.54,-14.308
3,Multiple Linear Regression,0.5344,0.5343,0.2804,0.2188,3.82,-14.853
4,Ridge Regression (alpha=1),0.5665,0.5664,0.2706,0.2068,3.61,-13.763
5,Random Forest (100 Trees),0.1522,0.1516,0.3784,0.2947,5.03,-27.882
6,XGBoost Default,0.2424,0.2421,0.3577,0.2620,4.46,-24.802
7,XGBoost Tuned (Nested Optuna),0.3225,0.3221,0.3383,0.2531,4.33,-22.079
8,"Anomali Ridge (iklim, dengan BULAN_BASAH)",0.9718,0.9718,0.0690,0.0595,1.02,0.039
9,"Anomali Ridge (iklim, tanpa BULAN_BASAH)",0.9709,0.9709,0.0701,0.0602,1.04,0.009


Skill vs Persistence = 1 - MSE_model / MSE_persistence (> 0 berarti lebih baik dari persistence).

Kesimpulan (otomatis):
1. Model dengan RMSE terendah: Anomali Ridge (iklim, dengan BULAN_BASAH) (RMSE = 0.069, MAPE = 1.02%).
2. Model berbasis fitur terbaik: Anomali Ridge (iklim, dengan BULAN_BASAH) (skill vs persistence = +0.039).
3. XGBoost Tuned (nested): RMSE = 0.3383, skill vs persistence = -22.079.
4. Model anomali iklim: dengan BULAN_BASAH skill = +0.039, tanpa BULAN_BASAH skill = +0.009.


## EKSPERIMEN 4: EVALUASI TEMPORAL BERKEADILAN (MATCHED TARGET YEARS 2023–2024)
Eksperimen ini menjawab langsung kritik tajam Reviewer 1 dan Reviewer 2 terkait perancuan evaluasi temporal (*confounded window comparison*). Seluruh skenario jendela waktu (SW-5 s.d. SW-8 dan Expanding Window) dievaluasi pada himpunan tahun target pengujian yang seragam persis (irisan 2023 dan 2024, total $N=54$ sampel uji), dilengkapi dengan analisis sensitivitas per tahun target (2020–2024), penghitungan *Pooled RMSE*, serta pembuktian kapasitas generalisasi model pasca-tuning Optuna antara SW-5 dan SW-8.

In [7]:
# =========================================================
# EKSPERIMEN 4: EVALUASI TEMPORAL BERKEADILAN (MATCHED TARGET YEARS)
# Menjawab Reviewer 1 & Reviewer 2 (Confounded Comparison)
# =========================================================

import numpy as np
import pandas as pd
import xgboost as xgb
from IPython.display import display
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error
)

MATCHED_TARGETS = [2023, 2024]
WINDOWS = [5, 6, 7, 8]

print("=== EKSPERIMEN 4: EVALUASI TEMPORAL BERKEADILAN (MATCHED TARGET YEARS) ===")
print(f"Skenario Uji: Evaluasi Setara pada Tahun Target {MATCHED_TARGETS} (Total N={27 * len(MATCHED_TARGETS)})\n")

# ---------------------------------------------------------
# 1. Evaluasi Performa per Tahun Target (2020-2024), termasuk persistence
# ---------------------------------------------------------
target_years = [2020, 2021, 2022, 2023, 2024]
results_by_year = []

for tyr in target_years:
    v_idx = df_master[df_master['YEAR_fitur'] == tyr - 1].index
    y_v = y.loc[v_idx]
    pred = df_master.loc[v_idx, 'PRODUKTIVITAS_T']
    results_by_year.append({'Window': 'Persistence', 'Target Year': tyr,
                            'R2': r2_score(y_v, pred), 'RMSE': np.sqrt(mean_squared_error(y_v, pred))})

for w in WINDOWS:
    for tyr in target_years:
        tf = tyr - 1
        start_tf = tf - w
        if start_tf < min(years_available):
            continue
        tr_years = list(range(start_tf, tf))
        t_idx = df_master[df_master['YEAR_fitur'].isin(tr_years)].index
        v_idx = df_master[df_master['YEAR_fitur'] == tf].index

        m = xgb.XGBRegressor(random_state=42).fit(X_processed.loc[t_idx], y.loc[t_idx])
        pred = m.predict(X_processed.loc[v_idx])
        y_v = y.loc[v_idx]

        results_by_year.append({
            'Window': f'SW-{w}',
            'Target Year': tyr,
            'R2': r2_score(y_v, pred),
            'RMSE': np.sqrt(mean_squared_error(y_v, pred)),
        })

df_by_year = pd.DataFrame(results_by_year)
row_order = ['Persistence'] + [f'SW-{w}' for w in WINDOWS]
df_pivot_r2 = df_by_year.pivot(index='Window', columns='Target Year', values='R2').reindex(row_order).round(4)
df_pivot_rmse = df_by_year.pivot(index='Window', columns='Target Year', values='RMSE').reindex(row_order).round(4)

print("======================================================================")
print("TABEL 1a. R² PER TAHUN TARGET (XGBOOST DEFAULT)")
print("======================================================================")
display(df_pivot_r2)
print("======================================================================")
print("TABEL 1b. RMSE (TON/HA) PER TAHUN TARGET (XGBOOST DEFAULT)")
print("======================================================================")
display(df_pivot_rmse)

# ---------------------------------------------------------
# 2. Evaluasi Setara / Matched Target Years (2023 & 2024)
# ---------------------------------------------------------
def matched_row(name, n_train, tr_years_fn, params):
    y_true_all, y_pred_all = [], []
    r2_list, rmse_list = [], []
    for tyr in MATCHED_TARGETS:
        tf = tyr - 1
        t_idx = df_master[df_master['YEAR_fitur'].isin(tr_years_fn(tf))].index
        v_idx = df_master[df_master['YEAR_fitur'] == tf].index
        m = xgb.XGBRegressor(**params).fit(X_processed.loc[t_idx], y.loc[t_idx])
        pred = m.predict(X_processed.loc[v_idx])
        y_v = y.loc[v_idx]
        r2_list.append(r2_score(y_v, pred))
        rmse_list.append(np.sqrt(mean_squared_error(y_v, pred)))
        y_true_all.extend(y_v.tolist())
        y_pred_all.extend(pred.tolist())
    y_true_all, y_pred_all = np.array(y_true_all), np.array(y_pred_all)
    return {
        'Skenario Split': name,
        'Data Latih (N)': n_train,
        'Data Uji (N)': len(y_true_all),
        'R2 (2023)': round(r2_list[0], 4),
        'R2 (2024)': round(r2_list[1], 4),
        'R2 (Mean Fold)': round(np.mean(r2_list), 4),
        'R2 (Pooled)': round(r2_score(y_true_all, y_pred_all), 4),
        'RMSE (Pooled)': round(np.sqrt(mean_squared_error(y_true_all, y_pred_all)), 4),
        'MAE': round(mean_absolute_error(y_true_all, y_pred_all), 4),
        'MAPE (%)': round(mean_absolute_percentage_error(y_true_all, y_pred_all) * 100, 2)
    }

matched_rows = [
    matched_row(f'True Sliding Window ({w} Thn)', w * 27, lambda tf, w=w: list(range(tf - w, tf)), {'random_state': 42})
    for w in WINDOWS
]
matched_rows.append(matched_row('Expanding Window', '216-243', lambda tf: [yr for yr in years_available if yr < tf], {'random_state': 42}))

df_matched = pd.DataFrame(matched_rows)
print("\n======================================================================")
print("TABEL 2. KOMPARASI EVALUASI SETARA (MATCHED TARGET YEARS: 2023-2024), XGBOOST DEFAULT")
print("======================================================================")
display(df_matched)

# ---------------------------------------------------------
# 3. Nested Optuna untuk SETIAP window (tiap window di-tuning dengan datanya sendiri)
# ---------------------------------------------------------
tuned_rows = []
for w in WINDOWS:
    _, df_pred_w = nested_walk_forward(w, test_feature_years=[t - 1 for t in MATCHED_TARGETS])
    tuned_rows.append({'Model & Skenario Jendela': f'SW-{w} Default', **summarize_predictions(df_pred_w, 'pred_default')})
    tuned_rows.append({'Model & Skenario Jendela': f'SW-{w} Tuned (Nested Optuna)', **summarize_predictions(df_pred_w, 'pred_tuned')})

df_tuned_comp = pd.DataFrame(tuned_rows)
print("\n======================================================================")
print("TABEL 3. DEFAULT VS NESTED OPTUNA PER WINDOW (MATCHED 2023-2024)")
print("======================================================================")
display(df_tuned_comp)

# ---------------------------------------------------------
# 4. Kesimpulan otomatis
# ---------------------------------------------------------
sw_only = df_matched[df_matched['Skenario Split'].str.startswith('True Sliding')]
winners = {
    'R2 (Mean Fold)': sw_only.loc[sw_only['R2 (Mean Fold)'].idxmax(), 'Skenario Split'],
    'RMSE (Pooled)': sw_only.loc[sw_only['RMSE (Pooled)'].idxmin(), 'Skenario Split'],
    'MAE': sw_only.loc[sw_only['MAE'].idxmin(), 'Skenario Split'],
    'MAPE (%)': sw_only.loc[sw_only['MAPE (%)'].idxmin(), 'Skenario Split'],
}
sw_years = df_pivot_r2.loc[[f'SW-{w}' for w in WINDOWS], MATCHED_TARGETS]
spread_window = (sw_years.max(axis=0) - sw_years.min(axis=0)).mean()
spread_year = (sw_years[MATCHED_TARGETS[0]] - sw_years[MATCHED_TARGETS[1]]).abs().mean()
tuned_only = df_tuned_comp[df_tuned_comp['Model & Skenario Jendela'].str.contains('Tuned')]
best_tuned = tuned_only.loc[tuned_only['RMSE (Ton/Ha, Pooled)'].idxmin(), 'Model & Skenario Jendela']

print("\nKesimpulan (otomatis):")
for metric, win in winners.items():
    print(f"- Window terbaik berdasarkan {metric}: {win}")
print(f"- Rata-rata selisih R² antar-window dalam tahun yang sama : {spread_window:.4f}")
print(f"- Rata-rata selisih R² antar-tahun (2023 vs 2024) per window: {spread_year:.4f}")
if spread_year > spread_window:
    print("  -> Variasi akibat TAHUN TARGET lebih besar daripada variasi akibat PANJANG WINDOW;")
    print("     kesimpulan pemilihan window perlu disampaikan dengan hati-hati (hanya 2 tahun uji).")
print(f"- Setelah nested tuning, RMSE terendah dicapai oleh: {best_tuned}")


=== EKSPERIMEN 4: EVALUASI TEMPORAL BERKEADILAN (MATCHED TARGET YEARS) ===
Skenario Uji: Evaluasi Setara pada Tahun Target [2023, 2024] (Total N=54)



TABEL 1a. R² PER TAHUN TARGET (XGBOOST DEFAULT)


Target Year,2020,2021,2022,2023,2024
Window,,,,,
Persistence,0.7310,0.6490,0.3116,0.9613,0.9800
SW-5,0.4245,-0.6237,0.2968,0.3100,0.0985
SW-6,NaN,-2.3036,0.2714,-0.4559,0.1715
SW-7,NaN,NaN,0.3364,0.3995,0.3425
SW-8,NaN,NaN,NaN,0.1097,0.3752


TABEL 1b. RMSE (TON/HA) PER TAHUN TARGET (XGBOOST DEFAULT)


Target Year,2020,2021,2022,2023,2024
Window,,,,,
Persistence,0.2138,0.1687,0.3462,0.0809,0.0581
SW-5,0.3128,0.3628,0.3499,0.3418,0.3895
SW-6,NaN,0.5175,0.3562,0.4964,0.3734
SW-7,NaN,NaN,0.3399,0.3188,0.3326
SW-8,NaN,NaN,NaN,0.3882,0.3243



TABEL 2. KOMPARASI EVALUASI SETARA (MATCHED TARGET YEARS: 2023-2024), XGBOOST DEFAULT


,Skenario Split,Data Latih (N),Data Uji (N),R2 (2023),R2 (2024),R2 (Mean Fold),R2 (Pooled),RMSE (Pooled),MAE,MAPE (%)
0,True Sliding Window (5 Thn),135,54,0.3100,0.0985,0.2042,0.2046,0.3664,0.2634,4.46
1,True Sliding Window (6 Thn),162,54,-0.4559,0.1715,-0.1422,-0.1430,0.4392,0.3181,5.42
2,True Sliding Window (7 Thn),189,54,0.3995,0.3425,0.3710,0.3711,0.3258,0.2286,3.90
3,True Sliding Window (8 Thn),216,54,0.1097,0.3752,0.2424,0.2421,0.3577,0.2620,4.46
4,Expanding Window,216-243,54,0.1097,0.3136,0.2116,0.2114,0.3649,0.2650,4.52



TABEL 3. DEFAULT VS NESTED OPTUNA PER WINDOW (MATCHED 2023-2024)


,Model & Skenario Jendela,R² (Mean Fold),R² (Pooled),"RMSE (Ton/Ha, Pooled)",MAE (Ton/Ha),MAPE (%)
0,SW-5 Default,0.2042,0.2046,0.3664,0.2634,4.46
1,SW-5 Tuned (Nested Optuna),0.2195,0.2193,0.3630,0.2589,4.39
2,SW-6 Default,-0.1422,-0.1430,0.4392,0.3181,5.42
3,SW-6 Tuned (Nested Optuna),0.1554,0.1550,0.3777,0.2863,4.88
4,SW-7 Default,0.3710,0.3711,0.3258,0.2286,3.90
5,SW-7 Tuned (Nested Optuna),0.3814,0.3812,0.3232,0.2356,4.04
6,SW-8 Default,0.2424,0.2421,0.3577,0.2620,4.46
7,SW-8 Tuned (Nested Optuna),0.3225,0.3221,0.3383,0.2531,4.33



Kesimpulan (otomatis):
- Window terbaik berdasarkan R2 (Mean Fold): True Sliding Window (7 Thn)
- Window terbaik berdasarkan RMSE (Pooled): True Sliding Window (7 Thn)
- Window terbaik berdasarkan MAE: True Sliding Window (7 Thn)
- Window terbaik berdasarkan MAPE (%): True Sliding Window (7 Thn)
- Rata-rata selisih R² antar-window dalam tahun yang sama : 0.5660
- Rata-rata selisih R² antar-tahun (2023 vs 2024) per window: 0.2903
- Setelah nested tuning, RMSE terendah dicapai oleh: SW-7 Tuned (Nested Optuna)


## EKSPERIMEN 5: MODEL USULAN - HYBRID PERSISTENCE + KOREKSI IKLIM
Prediksi produktivitas t+1 = produktivitas t + koreksi iklim. Koreksi iklim diestimasi dengan Ridge regression dari fitur agroklimat tahun t (alpha dipilih dengan validasi internal di dalam jendela latih). Model dievaluasi terhadap persistence (skill score), per window, per tahun target, dan dengan ablasi drop-one-feature.

In [8]:
# =========================================================
# EKSPERIMEN 5: MODEL USULAN - HYBRID PERSISTENCE + KOREKSI IKLIM
#   Prediksi Y_(t+1) = Y_t + f(iklim_t)
#   f = Ridge regression (fitur distandardisasi) yang memprediksi perubahan
#   produktivitas; alpha dipilih dengan validasi internal di dalam jendela latih.
# =========================================================

import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error

print("=== EKSPERIMEN 5: MODEL HYBRID (PERSISTENCE + KOREKSI IKLIM) ===")

HYB_W = 8

def hybrid_predict(train_years, test_feature_year, cols):
    t_idx = df_master[df_master['YEAR_fitur'].isin(train_years)].index
    v_idx = df_master[df_master['YEAR_fitur'] == test_feature_year].index
    alpha = choose_alpha(train_years, cols)
    model = fit_ridge_anomaly(t_idx, cols, alpha)
    pred = df_master.loc[v_idx, 'PRODUKTIVITAS_T'].values + model.predict(df_master.loc[v_idx, cols])
    return v_idx, pred, alpha

def hybrid_walk_forward(window, cols, test_feature_years=None):
    if test_feature_years is None:
        test_feature_years = years_available[window:]
    rows = []
    for tf in test_feature_years:
        tr_years = [yr for yr in years_available if tf - window <= yr < tf]
        v_idx, pred, alpha = hybrid_predict(tr_years, tf, cols)
        rows.append(pd.DataFrame({
            'WADMKC': df_master.loc[v_idx, 'WADMKC'].values,
            'YEAR_TARGET': tf + 1,
            'y_true': y.loc[v_idx].values,
            'pred_persist': df_master.loc[v_idx, 'PRODUKTIVITAS_T'].values,
            'pred_hybrid': pred,
            'alpha': alpha,
        }, index=v_idx))
    return pd.concat(rows)

def skill(df, col):
    return 1 - mean_squared_error(df['y_true'], df[col]) / mean_squared_error(df['y_true'], df['pred_persist'])

# ---------------------------------------------------------
# 1. Kinerja per window (seluruh tahun uji yang tersedia & matched 2023-2024)
# ---------------------------------------------------------
win_rows = []
hyb_by_window = {}
for wl in [5, 6, 7, 8]:
    d = hybrid_walk_forward(wl, ANOM_COLS)
    hyb_by_window[wl] = d
    dm = d[d['YEAR_TARGET'].isin(MATCHED_TARGETS)]
    win_rows.append({
        'Window': f'SW-{wl}',
        'Tahun Target Uji': f"{d['YEAR_TARGET'].min()}-{d['YEAR_TARGET'].max()}",
        'N Uji': len(d),
        'R² Hybrid': round(r2_score(d['y_true'], d['pred_hybrid']), 4),
        'R² Persistence': round(r2_score(d['y_true'], d['pred_persist']), 4),
        'RMSE Hybrid': round(np.sqrt(mean_squared_error(d['y_true'], d['pred_hybrid'])), 4),
        'RMSE Persistence': round(np.sqrt(mean_squared_error(d['y_true'], d['pred_persist'])), 4),
        'Skill (semua tahun)': round(skill(d, 'pred_hybrid'), 3),
        'Skill (matched 2023-2024)': round(skill(dm, 'pred_hybrid'), 3),
    })
df_hyb_windows = pd.DataFrame(win_rows)
print("\nTABEL 1. MODEL HYBRID VS PERSISTENCE PER WINDOW (PRODUKTIVITAS, TON/HA)")
display(df_hyb_windows)

# ---------------------------------------------------------
# 2. Rincian per tahun target (window terpanjang yang menguji 5 tahun: SW-5)
# ---------------------------------------------------------
year_rows = []
for wl in [5, HYB_W]:
    for yr, g in hyb_by_window[wl].groupby('YEAR_TARGET'):
        year_rows.append({
            'Window': f'SW-{wl}', 'Tahun Target': yr,
            'RMSE Hybrid': round(np.sqrt(mean_squared_error(g['y_true'], g['pred_hybrid'])), 4),
            'RMSE Persistence': round(np.sqrt(mean_squared_error(g['y_true'], g['pred_persist'])), 4),
            'Hybrid Lebih Baik': np.sqrt(mean_squared_error(g['y_true'], g['pred_hybrid']))
                                 < np.sqrt(mean_squared_error(g['y_true'], g['pred_persist'])),
            'Alpha Terpilih': g['alpha'].iloc[0],
        })
df_hyb_years = pd.DataFrame(year_rows)
print("\nTABEL 2. HYBRID VS PERSISTENCE PER TAHUN TARGET")
display(df_hyb_years)

# ---------------------------------------------------------
# 3. Metrik lengkap model usulan (SW-8) + uji perbedaan galat
# ---------------------------------------------------------
df_hyb_main = hyb_by_window[HYB_W]
abs_h = np.abs(df_hyb_main['y_true'] - df_hyb_main['pred_hybrid'])
abs_p = np.abs(df_hyb_main['y_true'] - df_hyb_main['pred_persist'])
w_stat, w_p = stats.wilcoxon(abs_h, abs_p)
RMSE_OOS_HYBRID = float(np.sqrt(mean_squared_error(df_hyb_main['y_true'], df_hyb_main['pred_hybrid'])))

df_hyb_metrics = pd.DataFrame([
    {'Model': f'Hybrid Persistence + Iklim (SW-{HYB_W})', **summarize_predictions(df_hyb_main, 'pred_hybrid')},
    {'Model': 'Naive Persistence (Y_t)', **summarize_predictions(df_hyb_main, 'pred_persist')},
])
print(f"\nTABEL 3. METRIK MODEL USULAN (SW-{HYB_W}, TARGET {sorted(df_hyb_main['YEAR_TARGET'].unique())})")
display(df_hyb_metrics)
print(f"Wilcoxon signed-rank pada |galat| per kecamatan (n = {len(abs_h)}): p = {w_p:.4f}")
print("Catatan: galat dalam satu tahun saling berkorelasi (iklim sama), sehingga p-value ini optimistis.")

# ---------------------------------------------------------
# 4. Ablasi drop-one-feature pada model hybrid (skill vs persistence)
# ---------------------------------------------------------
abl_rows = []
for drop in [None] + ANOM_COLS:
    cols = ANOM_COLS if drop is None else [c for c in ANOM_COLS if c != drop]
    d_all = pd.concat([hybrid_walk_forward(wl, cols) for wl in [HYB_W]])
    d5 = hybrid_walk_forward(5, cols)
    abl_rows.append({
        'Fitur Dihapus': 'Tidak ada (model lengkap)' if drop is None else drop,
        f'RMSE SW-{HYB_W}': round(np.sqrt(mean_squared_error(d_all['y_true'], d_all['pred_hybrid'])), 4),
        f'Skill SW-{HYB_W}': round(skill(d_all, 'pred_hybrid'), 3),
        'Skill SW-5 (2020-2024)': round(skill(d5, 'pred_hybrid'), 3),
    })
df_hyb_ablation = pd.DataFrame(abl_rows)
print("\nTABEL 4. ABLASI DROP-ONE-FEATURE PADA MODEL HYBRID")
display(df_hyb_ablation)

# ---------------------------------------------------------
# 5. Koefisien model (jendela latih terakhir) - interpretasi arah pengaruh
# ---------------------------------------------------------
final_alpha = choose_alpha(FINAL_TRAIN_YEARS, ANOM_COLS)
final_t_idx = df_master[df_master['YEAR_fitur'].isin(FINAL_TRAIN_YEARS)].index
hybrid_final_model = fit_ridge_anomaly(final_t_idx, ANOM_COLS, final_alpha)
df_hyb_coef = pd.DataFrame({
    'Fitur': ANOM_COLS,
    'Koefisien Terstandardisasi (ton/ha per 1 SD)': np.round(hybrid_final_model[-1].coef_, 4),
}).sort_values('Koefisien Terstandardisasi (ton/ha per 1 SD)', key=np.abs, ascending=False)
print(f"\nTABEL 5. KOEFISIEN MODEL HYBRID FINAL (jendela {min(FINAL_TRAIN_YEARS)}-{max(FINAL_TRAIN_YEARS)}, alpha = {final_alpha})")
display(df_hyb_coef)

# ---------------------------------------------------------
# 6. Kesimpulan otomatis
# ---------------------------------------------------------
full = df_hyb_ablation.iloc[0]
no_bb = df_hyb_ablation[df_hyb_ablation['Fitur Dihapus'] == 'BULAN_BASAH'].iloc[0]
n_win_pos = int((df_hyb_windows['Skill (semua tahun)'] > 0).sum())
sw5_years = df_hyb_years[df_hyb_years['Window'] == 'SW-5']
print("\nKesimpulan (otomatis):")
print(f"1. SW-{HYB_W}: R² hybrid = {df_hyb_metrics.loc[0, 'R² (Pooled)']}, RMSE = {RMSE_OOS_HYBRID:.4f} ton/ha, "
      f"skill vs persistence = {full[f'Skill SW-{HYB_W}']:+.3f}.")
print(f"2. Skill positif di {n_win_pos} dari {len(df_hyb_windows)} panjang window.")
print(f"3. Per tahun (SW-5): hybrid lebih baik dari persistence pada {int(sw5_years['Hybrid Lebih Baik'].sum())} "
      f"dari {len(sw5_years)} tahun target.")
print(f"4. Menghapus BULAN_BASAH mengubah skill SW-{HYB_W} dari {full[f'Skill SW-{HYB_W}']:+.3f} menjadi "
      f"{no_bb[f'Skill SW-{HYB_W}']:+.3f}; SW-5 dari {full['Skill SW-5 (2020-2024)']:+.3f} menjadi "
      f"{no_bb['Skill SW-5 (2020-2024)']:+.3f}.")


=== EKSPERIMEN 5: MODEL HYBRID (PERSISTENCE + KOREKSI IKLIM) ===



TABEL 1. MODEL HYBRID VS PERSISTENCE PER WINDOW (PRODUKTIVITAS, TON/HA)


,Window,Tahun Target Uji,N Uji,R² Hybrid,R² Persistence,RMSE Hybrid,RMSE Persistence,Skill (semua tahun),Skill (matched 2023-2024)
0,SW-5,2020-2024,135,0.7185,0.7338,0.2077,0.2020,-0.057,0.178
1,SW-6,2021-2024,108,0.7007,0.7346,0.2112,0.1989,-0.128,-0.185
2,SW-7,2022-2024,81,0.6690,0.7472,0.2380,0.2080,-0.309,0.107
3,SW-8,2023-2024,54,0.9718,0.9706,0.0690,0.0704,0.039,0.039



TABEL 2. HYBRID VS PERSISTENCE PER TAHUN TARGET


,Window,Tahun Target,RMSE Hybrid,RMSE Persistence,Hybrid Lebih Baik,Alpha Terpilih
0,SW-5,2020,0.2155,0.2138,False,1000
1,SW-5,2021,0.1660,0.1687,True,1000
2,SW-5,2022,0.3654,0.3462,False,1000
3,SW-5,2023,0.0691,0.0809,True,1000
4,SW-5,2024,0.0580,0.0581,True,1000
5,SW-8,2023,0.0772,0.0809,True,1000
6,SW-8,2024,0.0597,0.0581,False,1000



TABEL 3. METRIK MODEL USULAN (SW-8, TARGET [np.int64(2023), np.int64(2024)])


... (3 outputs left out)

,Fitur Dihapus,RMSE SW-8,Skill SW-8,Skill SW-5 (2020-2024)
0,Tidak ada (model lengkap),0.0690,0.039,-0.057
1,TOTAL_HUJAN,0.0698,0.017,-0.060
2,BULAN_BASAH,0.0701,0.009,-0.059
3,AVG_RADIASI,0.0647,0.155,-0.040
4,AVG_KELEMBABAN,0.0662,0.116,-0.029
5,MAX_SUHU_THN,0.0666,0.104,-0.030
6,MIN_SUHU_THN,0.2577,-12.392,-0.421
7,AVG_SUHU,0.1408,-2.999,-0.052
8,YEAR_fitur,0.0897,-0.621,-0.030



TABEL 5. KOEFISIEN MODEL HYBRID FINAL (jendela 2016-2023, alpha = 1000)


,Fitur,Koefisien Terstandardisasi (ton/ha per 1 SD)
7,YEAR_fitur,-0.0049
6,AVG_SUHU,-0.0044
4,MAX_SUHU_THN,-0.0040
1,BULAN_BASAH,-0.0030
0,TOTAL_HUJAN,0.0029
3,AVG_KELEMBABAN,0.0026
5,MIN_SUHU_THN,-0.0025
2,AVG_RADIASI,-0.0020



Kesimpulan (otomatis):
1. SW-8: R² hybrid = 0.9718, RMSE = 0.0690 ton/ha, skill vs persistence = +0.039.
2. Skill positif di 1 dari 4 panjang window.
3. Per tahun (SW-5): hybrid lebih baik dari persistence pada 3 dari 5 tahun target.
4. Menghapus BULAN_BASAH mengubah skill SW-8 dari +0.039 menjadi +0.009; SW-5 dari -0.057 menjadi -0.059.


## PROYEKSI PRODUKSI 2025
Model usulan: hybrid persistence + koreksi iklim (Eksperimen 5). Ketidakpastian memakai RMSE out-of-sample model hybrid dan asumsi galat produktivitas berkorelasi antar-kecamatan; XGBoost dan persistence murni ditampilkan sebagai pembanding.

In [9]:
# =========================================================
# STEP 10 : PREDIKSI PRODUKSI PADI 2025
# Model usulan: hybrid persistence + koreksi iklim (Eksperimen 5), dilatih pada
# jendela terakhir. XGBoost SW-8 & persistence murni ditampilkan sebagai pembanding.
# Galat produktivitas = RMSE out-of-sample model hybrid.
# Dilengkapi Geometric Mean, Delta Method, & Interval Ketidakpastian 95%
# =========================================================
print("10. Menghitung Prediksi Produksi 2025...")

X_pred_raw = df_input_2025[X_COLS + ['WADMKC']].copy()

X_pred = pd.get_dummies(
    X_pred_raw,
    columns=['WADMKC'],
    dtype=int
)

X_pred['YEAR_INDEX'] = 2024

X_pred = X_pred.reindex(
    columns=X_processed.columns,
    fill_value=0
)

final_train_idx = df_master[df_master['YEAR_fitur'].isin(FINAL_TRAIN_YEARS)].index
model_terbaik = xgb.XGBRegressor(**FINAL_PARAMS)
model_terbaik.fit(
    X_processed.loc[final_train_idx],
    y.loc[final_train_idx]
)

pred_produktivitas_xgb = model_terbaik.predict(X_pred)

# Model usulan: hybrid persistence + koreksi iklim (Eksperimen 5)
X_hyb_2025 = df_input_2025[X_COLS].copy()
X_hyb_2025['YEAR_fitur'] = 2024
pred_produktivitas_2025 = (
    df_input_2025['PRODUKTIVITAS'].values
    + hybrid_final_model.predict(X_hyb_2025[ANOM_COLS])
)

# RMSE produktivitas out-of-sample model hybrid (Eksperimen 5) - tidak diketik manual
rmse_model = RMSE_OOS_HYBRID

df_hasil_2025 = df_input_2025[['WADMKC']].copy().reset_index(drop=True)

df_hasil_2025['PREDIKSI_PRODUKTIVITAS']  = pred_produktivitas_2025
df_hasil_2025['PRODUKTIVITAS_2024']      = df_input_2025['PRODUKTIVITAS'].values
df_hasil_2025['LUAS_PANEN_2024']         = df_input_2025['LUAS_PANEN'].values
df_hasil_2025['RASIO_GM']                = df_input_2025['FAKTOR_KOREKSI'].values
df_hasil_2025['LUAS_PANEN_2025_EST']     = df_input_2025['LUAS_PANEN_2025_EST'].values
df_hasil_2025['PREDIKSI_PRODUKSI_TON']   = (
    pred_produktivitas_2025
    * df_hasil_2025['LUAS_PANEN_2025_EST'].values
)
df_hasil_2025['PRODUKSI_2024_AKTUAL']    = df_input_2025['PRODUKSI_PADI'].values
df_hasil_2025['DELTA_TON']               = (
    df_hasil_2025['PREDIKSI_PRODUKSI_TON']
    - df_hasil_2025['PRODUKSI_2024_AKTUAL']
)
df_hasil_2025['DELTA_PERSEN']            = (
    df_hasil_2025['DELTA_TON']
    / df_hasil_2025['PRODUKSI_2024_AKTUAL']
    * 100
).round(2)

# =========================================================
# PROPAGASI KETIDAKPASTIAN (DELTA METHOD)
# Var(P_i) = A_i^2 * Var(Y_i) + Y_i^2 * Var(A_i)
#   Var(Y_i) = RMSE_oos^2 ; Var(A_i) = (Luas_2024 * Std_Rasio)^2
# Tingkat kabupaten: galat produktivitas diasumsikan BERKORELASI PENUH antar-kecamatan
# (input iklim berasal dari grid yang sama), galat luas panen diasumsikan independen.
# Versi independen penuh ditampilkan sebagai pembanding (lebih sempit / optimistis).
# =========================================================
A_est = df_hasil_2025['LUAS_PANEN_2025_EST'].values
sigma_area_i = df_input_2025['LUAS_PANEN'].values * df_input_2025['STD_RASIO'].values
var_area_i = (pred_produktivitas_2025 ** 2) * (sigma_area_i ** 2)
var_prod_i = (A_est ** 2) * (rmse_model ** 2)
se_produksi_i = np.sqrt(var_prod_i + var_area_i)

df_hasil_2025['SE_PRODUKSI_TON'] = se_produksi_i
df_hasil_2025['CI_LOWER_95']     = df_hasil_2025['PREDIKSI_PRODUKSI_TON'] - 1.96 * se_produksi_i
df_hasil_2025['CI_UPPER_95']     = df_hasil_2025['PREDIKSI_PRODUKSI_TON'] + 1.96 * se_produksi_i

total_2025 = df_hasil_2025['PREDIKSI_PRODUKSI_TON'].sum()
total_2024 = df_hasil_2025['PRODUKSI_2024_AKTUAL'].sum()
se_total_kab = np.sqrt((A_est * rmse_model).sum() ** 2 + var_area_i.sum())
se_total_indep = np.sqrt(var_prod_i.sum() + var_area_i.sum())

ci_lower_kab_95 = total_2025 - 1.96 * se_total_kab
ci_upper_kab_95 = total_2025 + 1.96 * se_total_kab
ci_lower_kab_68 = total_2025 - se_total_kab
ci_upper_kab_68 = total_2025 + se_total_kab

# Pembanding: persistence murni & XGBoost (produktivitas x luas panen 2025 estimasi)
total_2025_persist = (df_hasil_2025['PRODUKTIVITAS_2024'] * A_est).sum()
total_2025_xgb = (pred_produktivitas_xgb * A_est).sum()

TARGET_ACUAN = 344243

def round_sig(x, base=1000):
    return int(round(x / base) * base)

print("\n========================================================")
print("  ESTIMASI PRODUKSI PADI BANYUMAS 2025")
print("  (Geometric Mean & Propagasi Ketidakpastian Delta Method)")
print("========================================================")
print(f"  RMSE produktivitas out-of-sample   : {rmse_model:.4f} ton/ha")
print(f"  Total Produksi 2024 (Aktual)       : {round_sig(total_2024):>10,} Ton")
print(f"  Estimasi Titik 2025 (dibulatkan)   : {round_sig(total_2025):>10,} Ton")
print(f"  Perubahan 2024 -> 2025             : {(total_2025-total_2024)/total_2024*100:+.1f}%")
print(f"  SE Gabungan (galat berkorelasi)    : {round_sig(se_total_kab, 100):>10,} Ton")
print(f"  Rentang 95% (galat berkorelasi)    : [{round_sig(ci_lower_kab_95):,} s.d. {round_sig(ci_upper_kab_95):,}] Ton")
print(f"  Rentang 95% (asumsi independen)    : [{round_sig(total_2025 - 1.96 * se_total_indep):,} s.d. "
      f"{round_sig(total_2025 + 1.96 * se_total_indep):,}] Ton")
print(f"  Pembanding persistence (Y_2024 x A_2025): {round_sig(total_2025_persist):,} Ton")
print(f"  Pembanding XGBoost SW-8 (FINAL_PARAMS)  : {round_sig(total_2025_xgb):,} Ton")
print(f"  Target Acuan                       : {TARGET_ACUAN:>10,} Ton")
print("========================================================")
if ci_lower_kab_95 <= total_2024 <= ci_upper_kab_95:
    print("  Catatan: produksi 2024 berada DI DALAM rentang 95%, sehingga perubahan 2024->2025")
    print("  tidak dapat dinyatakan sebagai penurunan/kenaikan yang meyakinkan.")
else:
    print("  Catatan: produksi 2024 berada di luar rentang 95%.")

print("\n  Rincian per Kecamatan (Presisi Wajar & Rentang 95% CI):")
df_tampil = df_hasil_2025[[
    'WADMKC',
    'PRODUKSI_2024_AKTUAL',
    'LUAS_PANEN_2025_EST',
    'PREDIKSI_PRODUKTIVITAS',
    'PREDIKSI_PRODUKSI_TON',
    'CI_LOWER_95',
    'CI_UPPER_95',
    'DELTA_PERSEN'
]].copy()

print(df_tampil.to_string(index=False, formatters={
    'PRODUKSI_2024_AKTUAL': lambda x: f"{x:,.0f}",
    'LUAS_PANEN_2025_EST': lambda x: f"{x:,.0f}",
    'PREDIKSI_PRODUKTIVITAS': lambda x: f"{x:.2f}",
    'PREDIKSI_PRODUKSI_TON': lambda x: f"{x:,.0f}",
    'CI_LOWER_95': lambda x: f"{x:,.0f}",
    'CI_UPPER_95': lambda x: f"{x:,.0f}",
    'DELTA_PERSEN': lambda x: f"{x:+.1f}%"
}))

# =========================================================
# STEP 11 : SIMPAN CSV
# =========================================================
df_hasil_2025.to_csv(
    "Prediksi_Produksi_Banyumas_2025.csv",
    index=False
)

print("\n  Hasil tersimpan: Prediksi_Produksi_Banyumas_2025.csv")


10. Menghitung Prediksi Produksi 2025...



  ESTIMASI PRODUKSI PADI BANYUMAS 2025
  (Geometric Mean & Propagasi Ketidakpastian Delta Method)
  RMSE produktivitas out-of-sample   : 0.0690 ton/ha
  Total Produksi 2024 (Aktual)       :    340,000 Ton
  Estimasi Titik 2025 (dibulatkan)   :    332,000 Ton
  Perubahan 2024 -> 2025             : -2.5%
  SE Gabungan (galat berkorelasi)    :      9,800 Ton
  Rentang 95% (galat berkorelasi)    : [313,000 s.d. 351,000] Ton
  Rentang 95% (asumsi independen)    : [314,000 s.d. 350,000] Ton
  Pembanding persistence (Y_2024 x A_2025): 331,000 Ton
  Pembanding XGBoost SW-8 (FINAL_PARAMS)  : 321,000 Ton
  Target Acuan                       :    344,243 Ton
  Catatan: produksi 2024 berada DI DALAM rentang 95%, sehingga perubahan 2024->2025
  tidak dapat dinyatakan sebagai penurunan/kenaikan yang meyakinkan.

  Rincian per Kecamatan (Presisi Wajar & Rentang 95% CI):
            WADMKC PRODUKSI_2024_AKTUAL LUAS_PANEN_2025_EST PREDIKSI_PRODUKTIVITAS PREDIKSI_PRODUKSI_TON CI_LOWER_95 CI_UPPER_95 DE

VISUALISASI

In [10]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

feature Importance

In [11]:
# =========================================================
# BOXPLOT PRODUKTIVITAS
# =========================================================
print("Menampilkan BoxPlot Produktivitas...")
plt.figure(figsize=(6, 5))
sns.boxplot(y=df_dataset['PRODUKTIVITAS'], color='skyblue')
plt.title('Boxplot Sebaran Data Produktivitas Padi', fontsize=12, fontweight='bold')
plt.ylabel('Ton/Hektar')
plt.tight_layout()
plt.savefig("boxplot_produktivitas.png", dpi=150)
plt.close()
print(" -> Gambar tersimpan: boxplot_produktivitas.png")

# =========================================================
# FEATURE IMPORTANCE & SHAP VALUE ANALYSIS
# Dekomposisi 35 Fitur: Spasial vs Agroklimat vs Temporal
# =========================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap

print("\nMenganalisis Feature Importance (Total Gain) & Nilai SHAP...")

# 1-2. Total Gain & SHAP OUT-OF-SAMPLE:
#      satu model XGBoost default per fold SW-8; SHAP dihitung pada tahun uji fold tersebut
#      (bukan pada data latih), gain dijumlahkan antar-fold.
import xgboost as xgb
SHAP_W = 8
feature_names = list(X_processed.columns)
importance_complete = {f: 0.0 for f in feature_names}
shap_blocks, data_blocks = [], []
for start_idx in range(len(years_available) - SHAP_W):
    tr_years = years_available[start_idx : start_idx + SHAP_W]
    te_year = years_available[start_idx + SHAP_W]
    t_idx = df_master[df_master['YEAR_fitur'].isin(tr_years)].index
    v_idx = df_master[df_master['YEAR_fitur'] == te_year].index
    m_fold = xgb.XGBRegressor(random_state=42).fit(X_processed.loc[t_idx], y.loc[t_idx])
    for key, value in m_fold.get_booster().get_score(importance_type="total_gain").items():
        if key in importance_complete:
            importance_complete[key] += value
    shap_blocks.append(shap.TreeExplainer(m_fold).shap_values(X_processed.loc[v_idx]))
    data_blocks.append(X_processed.loc[v_idx])

df_gain = pd.DataFrame(list(importance_complete.items()), columns=['feature', 'total_gain'])
df_gain['gain_pct'] = df_gain['total_gain'] / df_gain['total_gain'].sum() * 100

shap_values = np.vstack(shap_blocks)
X_shap = pd.concat(data_blocks)
mean_abs_shap = np.abs(shap_values).mean(axis=0)

df_shap = pd.DataFrame({
    'feature': feature_names,
    'mean_abs_shap': mean_abs_shap
})
df_shap['shap_pct'] = df_shap['mean_abs_shap'] / df_shap['mean_abs_shap'].sum() * 100

# 3. Pengelompokan Makro (Grouped Decomposition)
sub_gain = df_gain[df_gain['feature'].str.startswith('WADMKC_')]['gain_pct'].sum()
agro_gain = df_gain[df_gain['feature'].isin(X_COLS)]['gain_pct'].sum()
year_gain = df_gain[df_gain['feature'] == 'YEAR_INDEX']['gain_pct'].sum()

sub_shap = df_shap[df_shap['feature'].str.startswith('WADMKC_')]['shap_pct'].sum()
agro_shap = df_shap[df_shap['feature'].isin(X_COLS)]['shap_pct'].sum()
year_shap = df_shap[df_shap['feature'] == 'YEAR_INDEX']['shap_pct'].sum()

sub_sum_shap = df_shap[df_shap['feature'].str.startswith('WADMKC_')]['mean_abs_shap'].sum()
agro_sum_shap = df_shap[df_shap['feature'].isin(X_COLS)]['mean_abs_shap'].sum()
year_sum_shap = df_shap[df_shap['feature'] == 'YEAR_INDEX']['mean_abs_shap'].sum()

df_grouped = pd.DataFrame([
    {
        'Kelompok Fitur': 'Identitas Kecamatan (27 Variabel Dummy)',
        'Peran Agronomi': 'Baseline Spasial (Tanah, Topografi, & Irigasi Lokal)',
        'XGBoost Gain (%)': round(sub_gain, 2),
        'Mean |SHAP|': round(sub_sum_shap, 4),
        'Kontribusi SHAP (%)': round(sub_shap, 2)
    },
    {
        'Kelompok Fitur': 'Fitur Agroklimat (7 Variabel Cuaca)',
        'Peran Agronomi': 'Sinyal Dinamis Antartahun (Hujan, Suhu, Radiasi)',
        'XGBoost Gain (%)': round(agro_gain, 2),
        'Mean |SHAP|': round(agro_sum_shap, 4),
        'Kontribusi SHAP (%)': round(agro_shap, 2)
    },
    {
        'Kelompok Fitur': 'Tren Waktu Makro (YEAR_INDEX)',
        'Peran Agronomi': 'Tren Teknologi Pertanian & Varietas Benih',
        'XGBoost Gain (%)': round(year_gain, 2),
        'Mean |SHAP|': round(year_sum_shap, 4),
        'Kontribusi SHAP (%)': round(year_shap, 2)
    }
])

print("\n" + "=" * 80)
print("TABEL 1. DEKOMPOSISI KONTRIBUSI FITUR MAKRO (SPASIAL VS IKLIM VS TEMPORAL)")
print("=" * 80)
display(df_grouped)

# 4. Rincian Fitur Agroklimat & Temporal
selected_features = [
    'TOTAL_HUJAN', 'BULAN_BASAH', 'AVG_RADIASI', 'AVG_KELEMBABAN',
    'MAX_SUHU_THN', 'MIN_SUHU_THN', 'AVG_SUHU', 'YEAR_INDEX'
]
display_names = {
    'TOTAL_HUJAN': 'Total Rainfall',
    'BULAN_BASAH': 'Wet Months',
    'AVG_RADIASI': 'Solar Radiation',
    'AVG_KELEMBABAN': 'Humidity',
    'MAX_SUHU_THN': 'Max Temperature',
    'MIN_SUHU_THN': 'Min Temperature',
    'AVG_SUHU': 'Mean Temperature',
    'YEAR_INDEX': 'Year Index'
}

df_detail = pd.DataFrame({
    'FEATURE': selected_features,
    'DISPLAY_NAME': [display_names[f] for f in selected_features],
    'GAIN': [importance_complete.get(f, 0.0) for f in selected_features],
    'MEAN_SHAP': [df_shap[df_shap['feature'] == f]['mean_abs_shap'].values[0] for f in selected_features]
})
df_detail['RELATIVE_GAIN'] = df_detail['GAIN'] / df_detail['GAIN'].sum() * 100
df_detail['RELATIVE_SHAP'] = df_detail['MEAN_SHAP'] / df_detail['MEAN_SHAP'].sum() * 100
df_detail = df_detail.sort_values('RELATIVE_SHAP', ascending=True).reset_index(drop=True)

print("\n" + "=" * 80)
print("TABEL 2. KONTRIBUSI RELATIF FITUR AGROKLIMAT & TEMPORAL")
print("=" * 80)
display(df_detail[['DISPLAY_NAME', 'RELATIVE_GAIN', 'RELATIVE_SHAP']].sort_values('RELATIVE_SHAP', ascending=False).round(2))

rank_shap = df_detail.sort_values('RELATIVE_SHAP', ascending=False).reset_index(drop=True)
bb_shap_rank = int(rank_shap.index[rank_shap['FEATURE'] == 'BULAN_BASAH'][0]) + 1
rank_gain = df_detail.sort_values('RELATIVE_GAIN', ascending=False).reset_index(drop=True)
bb_gain_rank = int(rank_gain.index[rank_gain['FEATURE'] == 'BULAN_BASAH'][0]) + 1
bb_shap_share = df_detail.loc[df_detail['FEATURE'] == 'BULAN_BASAH', 'RELATIVE_SHAP'].iloc[0]
print(f"\nWet Months: peringkat {bb_gain_rank} (gain) dan {bb_shap_rank} (SHAP) dari {len(df_detail)} fitur dinamis; "
      f"share SHAP = {bb_shap_share:.2f}%.")
print(f"Identitas kecamatan = {sub_shap:.2f}% SHAP, agroklimat = {agro_shap:.2f}%, YEAR_INDEX = {year_shap:.2f}% "
      f"(dihitung pada {len(X_shap)} baris uji out-of-sample).")

# 5. Visualisasi Dual-Panel Figure
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

groups = [
    'Subdistrict Identity\n(27 Spatial Dummies)',
    'Agroclimatic Features\n(7 Weather Signals)',
    'Macro Temporal Trend\n(Year Index)'
]
gain_vals = [sub_gain, agro_gain, year_gain]
shap_vals = [sub_shap, agro_shap, year_shap]

y_pos = np.arange(len(groups))
height = 0.35

rects1 = ax1.barh(y_pos - height/2, shap_vals, height, label='Mean |SHAP| (%)', color='#2b5c8f', edgecolor='black', linewidth=0.5)
rects2 = ax1.barh(y_pos + height/2, gain_vals, height, label='Total Gain (%)', color='#41b6c4', edgecolor='black', linewidth=0.5)

ax1.set_yticks(y_pos)
ax1.set_yticklabels(groups, fontsize=9.5, fontweight='bold')
ax1.set_xlabel('Relative Contribution (%)', fontsize=10)
ax1.set_title('(a) Macro Decomposition of Model Features', fontsize=11, fontweight='bold', pad=10)
ax1.grid(axis='x', linestyle='--', alpha=0.5)
ax1.set_xlim(0, 95)
ax1.legend(loc='lower right', frameon=True)

for bar in rects1:
    w = bar.get_width()
    ax1.text(w + 1.2, bar.get_y() + bar.get_height()/2, f'{w:.1f}%', va='center', fontsize=8.5, color='#1e3d59', fontweight='bold')
for bar in rects2:
    w = bar.get_width()
    ax1.text(w + 1.2, bar.get_y() + bar.get_height()/2, f'{w:.1f}%', va='center', fontsize=8.5, color='#176873')

y_pos2 = np.arange(len(df_detail))
bars_det = ax2.barh(y_pos2, df_detail['RELATIVE_SHAP'], height=0.6, color='#e67e22', edgecolor='black', linewidth=0.5)
ax2.set_yticks(y_pos2)
ax2.set_yticklabels(df_detail['DISPLAY_NAME'], fontsize=9.5)
ax2.set_xlabel('Relative Dynamic SHAP Contribution (%)', fontsize=10)
ax2.set_title('(b) Dynamic Agroclimatic & Temporal Importance', fontsize=11, fontweight='bold', pad=10)
ax2.grid(axis='x', linestyle='--', alpha=0.5)
ax2.set_xlim(0, max(df_detail['RELATIVE_SHAP']) * 1.25)

for bar in bars_det:
    w = bar.get_width()
    ax2.text(w + 0.6, bar.get_y() + bar.get_height()/2, f'{w:.1f}%', va='center', fontsize=8.5, fontweight='bold')

plt.tight_layout()
plt.savefig("feature_importance_agroclimatic.png", dpi=300, bbox_inches='tight')
plt.show()

print("\n -> Gambar tersimpan: feature_importance_agroclimatic.png")

# 6. Simpan SHAP Beeswarm Summary Plot
indices = [feature_names.index(f) for f in selected_features]
shap_sub = shap.Explanation(
    values=shap_values[:, indices],
    data=X_shap[selected_features].values,
    feature_names=[display_names[f] for f in selected_features]
)

plt.figure(figsize=(8, 5))
shap.plots.beeswarm(shap_sub, show=False, max_display=8)
plt.title('SHAP Summary Plot: Agroclimatic & Temporal Features', fontsize=11, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig("shap_summary_agroclimatic.png", dpi=300, bbox_inches='tight')
plt.show()
print(" -> Gambar tersimpan: shap_summary_agroclimatic.png")


Menampilkan BoxPlot Produktivitas...


 -> Gambar tersimpan: boxplot_produktivitas.png



Menganalisis Feature Importance (Total Gain) & Nilai SHAP...



TABEL 1. DEKOMPOSISI KONTRIBUSI FITUR MAKRO (SPASIAL VS IKLIM VS TEMPORAL)


,Kelompok Fitur,Peran Agronomi,XGBoost Gain (%),Mean |SHAP|,Kontribusi SHAP (%)
0,Identitas Kecamatan (27 Variabel Dummy),"Baseline Spasial (Tanah, Topografi, & Irigasi ...",64.40,0.4026,55.790001
1,Fitur Agroklimat (7 Variabel Cuaca),"Sinyal Dinamis Antartahun (Hujan, Suhu, Radiasi)",30.37,0.2823,39.130001
2,Tren Waktu Makro (YEAR_INDEX),Tren Teknologi Pertanian & Varietas Benih,5.23,0.0366,5.080000



TABEL 2. KONTRIBUSI RELATIF FITUR AGROKLIMAT & TEMPORAL


... (2 outputs left out)

C:\Users\acer\AppData\Local\Temp\ipykernel_16572\922608896.py:188: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



 -> Gambar tersimpan: feature_importance_agroclimatic.png


 -> Gambar tersimpan: shap_summary_agroclimatic.png


C:\Users\acer\AppData\Local\Temp\ipykernel_16572\922608896.py:205: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


HeatMap Korelasi

In [12]:
# =========================================================
# HEATMAP KORELASI
# =========================================================
print("Menampilkan Heatmap Korelasi...")

corr_cols = [
    'TOTAL_HUJAN',
    'BULAN_BASAH',
    'AVG_RADIASI',
    'AVG_KELEMBABAN',
    'MAX_SUHU_THN',
    'MIN_SUHU_THN',
    'AVG_SUHU',
    'PRODUKTIVITAS'
]

corr_matrix = df_dataset[corr_cols].corr()

plt.figure(figsize=(11, 8))

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='RdYlBu',
    vmin=-1,
    vmax=1,
    center=0,
    linewidths=0.7,
    square=True
)

plt.title(
    'Heatmap Korelasi Agroklimat vs Produktivitas',
    fontsize=15,
    fontweight='bold'
)

plt.xticks(rotation=35, ha='right')
plt.yticks(rotation=0)

plt.tight_layout()

plt.savefig(
    "heatmap_korelasi_agroklimat.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

Menampilkan Heatmap Korelasi...


C:\Users\acer\AppData\Local\Temp\ipykernel_16572\1155616295.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Peta Prediksi Produksi 2025

In [13]:
# =========================================================
# VISUALISASI PETA
# =========================================================
print("Membangun Peta Prediksi...")

gdf_peta = gdf_banyumas.merge(
    df_hasil_2025[
        ['WADMKC', 'PREDIKSI_PRODUKSI_TON']
    ],
    on='WADMKC',
    how='left'
)

fig, ax = plt.subplots(
    1,
    1,
    figsize=(14, 10)
)

gdf_peta.plot(
    column='PREDIKSI_PRODUKSI_TON',
    cmap='YlGn',
    linewidth=0.8,
    ax=ax,
    edgecolor='0.5',
    legend=True,
    legend_kwds={
        'label': '2025 Rice Production Forecast (Tons)',
        'orientation': 'horizontal',
        'shrink': 0.6,
        'format': mticker.FuncFormatter(
            lambda x, _: f"{x:,.0f}"
        )
    }
)

ax.set_title(
    f"2025 Paddy Production Map\n",
    fontsize=13,
    fontweight='bold'
)

ax.axis('off')

plt.tight_layout()

plt.savefig(
    "peta_prediksi_2025.png",
    dpi=150,
    bbox_inches='tight'
)

plt.show()

Membangun Peta Prediksi...


C:\Users\acer\AppData\Local\Temp\ipykernel_16572\966921362.py:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
# =========================================================
# VISUALISASI FORECASTING (LINE CHART)
# Dilengkapi Rentang Ketidakpastian 95% (Error Bar)
# =========================================================
print("Membangun Grafik Tren Produksi Historis & Prediksi 2025...")

# Agregasi data historis tahunan dari dataset master
df_historical_annual = df_dataset.groupby('YEAR')['PRODUKSI_PADI'].sum().reset_index()
df_historical_annual.rename(columns={'PRODUKSI_PADI': 'PRODUKSI_TON'}, inplace=True)
df_historical_annual['STATUS'] = 'Aktual'

# Ambil total prediksi 2025
total_prediksi_2025 = df_hasil_2025['PREDIKSI_PRODUKSI_TON'].sum()
df_pred_2025 = pd.DataFrame({
    'YEAR': [2025],
    'PRODUKSI_TON': [total_prediksi_2025],
    'STATUS': ['Prediksi 2025']
})

# Gabungkan untuk plotting kontinu
df_plot = pd.concat([df_historical_annual, df_pred_2025], ignore_index=True).sort_values('YEAR')

# Buat plot
plt.figure(figsize=(12, 6))

# Plot garis aktual
df_aktual = df_plot[df_plot['STATUS'] == 'Aktual']
plt.plot(df_aktual['YEAR'], df_aktual['PRODUKSI_TON'], marker='o', color='#1f77b4', linewidth=2.5, label='Produksi Aktual (Historis)')

# Plot garis penghubung dari tahun terakhir aktual ke tahun prediksi 2025
df_transition = df_plot[df_plot['YEAR'].isin([df_aktual['YEAR'].max(), 2025])]
plt.plot(df_transition['YEAR'], df_transition['PRODUKSI_TON'], linestyle='--', color='#ff7f0e', linewidth=2.5, label='Proyeksi ke 2025')

# Plot titik prediksi 2025 dengan rentang ketidakpastian 95% (error bar)
plt.errorbar(
    2025, total_prediksi_2025,
    yerr=1.96 * se_total_kab,
    fmt='o', color='red', ecolor='#d62728', elinewidth=2.5, capsize=6, capthick=2,
    zorder=6, label=f'Prediksi 2025 (95% CI: {ci_lower_kab_95:,.0f}–{ci_upper_kab_95:,.0f} Ton)'
)

# Tambahkan garis horizontal target acuan
plt.axhline(y=TARGET_ACUAN, color='green', linestyle=':', linewidth=2, label=f'Target Acuan ({TARGET_ACUAN:,} Ton)')

# Teks nilai di atas titik-titik beberapa tahun terakhir dan prediksi
for idx, row in df_plot.iterrows():
    if row['YEAR'] >= 2020:
        if row['YEAR'] == 2025:
            text_str = f"{round(row['PRODUKSI_TON'], -3):,.0f} T\n[95% CI: {round(ci_lower_kab_95, -3):,.0f}–{round(ci_upper_kab_95, -3):,.0f}]"
            plt.annotate(
                text_str,
                (row['YEAR'], row['PRODUKSI_TON']),
                textcoords="offset points",
                xytext=(0, 15),
                ha='center',
                fontsize=9,
                fontweight='bold',
                color='red'
            )
        else:
            plt.annotate(
                f"{row['PRODUKSI_TON']:,.0f} T",
                (row['YEAR'], row['PRODUKSI_TON']),
                textcoords="offset points",
                xytext=(0, 10),
                ha='center',
                fontsize=9
            )

# Dekorasi Chart
plt.title('Tren Tahunan Produksi Padi Kabupaten Banyumas & Estimasi 2025 (dengan Interval Ketidakpastian 95%)', fontsize=13, fontweight='bold', pad=15)
plt.xlabel('Tahun', fontsize=12)
plt.ylabel('Total Produksi (Ton)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.xticks(df_plot['YEAR'])
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.legend(loc='lower left', frameon=True, shadow=True)
plt.tight_layout()

# Simpan Gambar
plt.savefig("tren_forecasting_produksi_2025.png", dpi=150, bbox_inches='tight')
plt.show()

print(" -> Gambar tersimpan: tren_forecasting_produksi_2025.png")


C:\Users\acer\AppData\Local\Temp\ipykernel_16572\2016110036.py:9: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_historical_annual.rename(columns={'PRODUKSI_PADI': 'PRODUKSI_TON'}, inplace=True)


Membangun Grafik Tren Produksi Historis & Prediksi 2025...


 -> Gambar tersimpan: tren_forecasting_produksi_2025.png


C:\Users\acer\AppData\Local\Temp\ipykernel_16572\2016110036.py:82: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np
import pandas as pd

# 1. Definisikan konstanta warna dan variabel pendukung
GHOST_COLOR = '#d1d5db'  # Abu-abu muda transparan untuk ghost lines
MAIN_COLOR = '#1f77b4'   # Biru untuk historis kecamatan target
FORECAST_COLOR = '#ff7f0e' # Oranye untuk prediksi 2025

all_kecamatan = df_dataset['WADMKC'].unique()

# 2. Kelompokkan kecamatan ke dalam 3 Klaster berdasarkan Prediksi Produksi 2025
df_sort = df_hasil_2025.sort_values(by='PREDIKSI_PRODUKSI_TON', ascending=False).reset_index(drop=True)

# Bagi rata menjadi 3 kelompok (masing-masing 9 kecamatan)
klaster_tinggi = list(df_sort.loc[0:8, 'WADMKC'])
klaster_sedang = list(df_sort.loc[9:17, 'WADMKC'])
klaster_rendah = list(df_sort.loc[18:26, 'WADMKC'])

klasters = {
    'Klaster Produksi Tinggi': klaster_tinggi,
    'Klaster Produksi Sedang': klaster_sedang,
    'Klaster Produksi Rendah': klaster_rendah
}

KLASTER_DESKRIPSI = {
    'Klaster Produksi Tinggi': '> 18.000 Ton',
    'Klaster Produksi Sedang': '8.000 – 18.000 Ton',
    'Klaster Produksi Rendah': '< 8.000 Ton'
}

# 3. Loop Visualisasi Small Multiples per Klaster
for klaster_name, kec_list in klasters.items():
    kec_sorted = sorted(kec_list)
    fig, axes = plt.subplots(3, 3, figsize=(15, 12), sharex=True, sharey=True)
    axes_flat = axes.flatten()

    desc_ambang = KLASTER_DESKRIPSI.get(klaster_name, '')

    fig.suptitle(
        f"Tren & Proyeksi Produksi Padi (2014–2025) - {klaster_name} ({desc_ambang})",
        fontsize=15,
        fontweight='bold',
        y=0.98
    )

    for i, target_kec in enumerate(kec_sorted):
        ax = axes_flat[i]

        # --- A. GHOST LINES DARI SELURUH 27 KECAMATAN ---
        for other_kec in all_kecamatan:
            if other_kec == target_kec:
                continue
            df_other = df_dataset[df_dataset['WADMKC'] == other_kec][['YEAR', 'PRODUKSI_PADI']].sort_values('YEAR')
            pred_other = df_hasil_2025[df_hasil_2025['WADMKC'] == other_kec]['PREDIKSI_PRODUKSI_TON'].values[0]

            # Plot garis abu-abu tipis sebagai latar belakang pembanding
            ax.plot(df_other['YEAR'], df_other['PRODUKSI_PADI'],
                    color=GHOST_COLOR, linewidth=0.7, alpha=0.35, zorder=1)
            ax.plot([df_other['YEAR'].iloc[-1], 2025], [df_other['PRODUKSI_PADI'].iloc[-1], pred_other],
                    linestyle=':', color=GHOST_COLOR, linewidth=0.6, alpha=0.35, zorder=1)

        # --- B. TARGET KECAMATAN UTAMA ---
        df_target_plot = df_dataset[df_dataset['WADMKC'] == target_kec][['YEAR', 'PRODUKSI_PADI']].sort_values('YEAR')
        pred_target = df_hasil_2025[df_hasil_2025['WADMKC'] == target_kec]['PREDIKSI_PRODUKSI_TON'].values[0]
        last_yr = df_target_plot['YEAR'].iloc[-1]
        last_val = df_target_plot['PRODUKSI_PADI'].iloc[-1]

        # Plot Data Historis Utama
        ax.plot(df_target_plot['YEAR'], df_target_plot['PRODUKSI_PADI'],
                marker='o', markersize=4.5, color=MAIN_COLOR, linewidth=2.2, zorder=3, label='Historis')

        # Plot Garis Proyeksi ke 2025
        ax.plot([last_yr, 2025], [last_val, pred_target],
                linestyle='--', color=FORECAST_COLOR, linewidth=2.0, zorder=3, label='Prediksi 2025')
        ax.scatter(2025, pred_target, color=FORECAST_COLOR, s=45, zorder=4)

        # Label angka prediksi di atas titik 2025
        ax.annotate(f"{pred_target:,.0f} T",
                    (2025, pred_target),
                    textcoords="offset points", xytext=(0, 7), ha='center',
                    fontsize=8.5, fontweight='bold', color=FORECAST_COLOR)

        # Atur estetika subplot
        ax.set_title(target_kec, fontsize=11, fontweight='bold', pad=6, loc='left', color='#24292f')
        ax.grid(True, linestyle=':', alpha=0.5)
        ax.set_xticks([2014, 2017, 2020, 2023, 2025])
        ax.set_xlim(2013.3, 2026.2)
        ax.tick_params(axis='both', which='major', labelsize=8.5)
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f" {x:,.0f}"))

        if i % 3 == 0:
            ax.set_ylabel("Produksi (Ton)", fontsize=9.5)
        if i >= 6:
            ax.set_xlabel("Tahun", fontsize=9.5)
        if i == 0:
            ax.legend(loc='lower left', fontsize=7.5, frameon=True, framealpha=0.85)

    # Hapus subplot sisa jika jumlah kecamatan kurang dari 9
    for j in range(len(kec_sorted), 9):
        fig.delaxes(axes_flat[j])

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    file_name = f"small_multiples_globalghost_{klaster_name.lower().replace(' ', '_')}.png"
    plt.savefig(file_name, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"-> Berhasil menyimpan file visualisasi klaster: {file_name}")


C:\Users\acer\AppData\Local\Temp\ipykernel_16572\1835514808.py:108: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


-> Berhasil menyimpan file visualisasi klaster: small_multiples_globalghost_klaster_produksi_tinggi.png


C:\Users\acer\AppData\Local\Temp\ipykernel_16572\1835514808.py:108: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


-> Berhasil menyimpan file visualisasi klaster: small_multiples_globalghost_klaster_produksi_sedang.png


-> Berhasil menyimpan file visualisasi klaster: small_multiples_globalghost_klaster_produksi_rendah.png


C:\Users\acer\AppData\Local\Temp\ipykernel_16572\1835514808.py:108: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
# =========================================================
# ACTUAL VS PREDICTED - BEST TRUE SLIDING WINDOW
# Evaluasi pada Produktivitas (ton/ha) & Produksi (ton)
# =========================================================

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error
)

print("Membangun Actual vs. Predicted untuk skenario terbaik...")

# ---------------------------------------------------------
# 1. Ambil ukuran window dari skenario terbaik
# ---------------------------------------------------------
if "True Sliding Window" not in rasio_terbaik:
    raise ValueError(
        f"Skenario terbaik bukan True Sliding Window: {rasio_terbaik}"
    )

match = re.search(r"(\d+)\s*Thn", rasio_terbaik)

if match is None:
    raise ValueError(
        f"Ukuran window tidak dapat dibaca dari: {rasio_terbaik}"
    )

BEST_WINDOW = int(match.group(1))

print(f"Best window: {BEST_WINDOW} tahun")

# ---------------------------------------------------------
# 2. Walk-forward ulang dan simpan seluruh prediksi test
# ---------------------------------------------------------
actual_prodv_all = []
predicted_prodv_all = []
actual_ton_all = []
predicted_ton_all = []
district_all = []
target_year_all = []

for start_idx in range(len(years_available) - BEST_WINDOW):

    train_years = years_available[
        start_idx : start_idx + BEST_WINDOW
    ]

    test_year = years_available[
        start_idx + BEST_WINDOW
    ]

    train_idx = df_master[
        df_master["YEAR_fitur"].isin(train_years)
    ].index

    test_idx = df_master[
        df_master["YEAR_fitur"] == test_year
    ].index

    if len(train_idx) == 0 or len(test_idx) == 0:
        continue

    X_train = X_processed.loc[train_idx]
    X_test = X_processed.loc[test_idx]

    y_train = y.loc[train_idx]
    y_test = y.loc[test_idx]

    # Model baru untuk setiap fold
    model_fold = xgb.XGBRegressor(
        random_state=42
    )

    model_fold.fit(
        X_train,
        y_train
    )

    # Prediksi produktivitas
    pred_productivity = model_fold.predict(
        X_test
    )

    # Konversi ke produksi (ton)
    harvest_area = df_master.loc[
        test_idx,
        "LUAS_PANEN"
    ].values

    actual_ton = (
        y_test.values * harvest_area
    )

    predicted_ton = (
        pred_productivity * harvest_area
    )

    # Simpan produktivitas
    actual_prodv_all.extend(y_test.values)
    predicted_prodv_all.extend(pred_productivity)

    # Simpan produksi
    actual_ton_all.extend(actual_ton)
    predicted_ton_all.extend(predicted_ton)

    district_all.extend(
        df_master.loc[
            test_idx,
            "WADMKC"
        ].values
    )

    target_year_all.extend(
        df_master.loc[
            test_idx,
            "YEAR_TARGET"
        ].values
    )

# ---------------------------------------------------------
# 3. Simpan hasil seluruh fold ke DataFrame
# ---------------------------------------------------------
df_actual_pred_best = pd.DataFrame({
    "KECAMATAN": district_all,
    "YEAR_TARGET": target_year_all,
    "ACTUAL_PRODUCTIVITY": actual_prodv_all,
    "PREDICTED_PRODUCTIVITY": predicted_prodv_all,
    "ACTUAL_PRODUCTION": actual_ton_all,
    "PREDICTED_PRODUCTION": predicted_ton_all
})

# ---------------------------------------------------------
# 4. Hitung pooled metrics
# ---------------------------------------------------------
# A. PRODUKTIVITAS (ton/ha)
r2_prodv_pooled = r2_score(
    df_actual_pred_best["ACTUAL_PRODUCTIVITY"],
    df_actual_pred_best["PREDICTED_PRODUCTIVITY"]
)
rmse_prodv_pooled = np.sqrt(
    mean_squared_error(
        df_actual_pred_best["ACTUAL_PRODUCTIVITY"],
        df_actual_pred_best["PREDICTED_PRODUCTIVITY"]
    )
)
mae_prodv_pooled = mean_absolute_error(
    df_actual_pred_best["ACTUAL_PRODUCTIVITY"],
    df_actual_pred_best["PREDICTED_PRODUCTIVITY"]
)
mape_prodv_pooled = (
    mean_absolute_percentage_error(
        df_actual_pred_best["ACTUAL_PRODUCTIVITY"],
        df_actual_pred_best["PREDICTED_PRODUCTIVITY"]
    ) * 100
)

# B. PRODUKSI (ton)
r2_ton_pooled = r2_score(
    df_actual_pred_best["ACTUAL_PRODUCTION"],
    df_actual_pred_best["PREDICTED_PRODUCTION"]
)
rmse_ton_pooled = np.sqrt(
    mean_squared_error(
        df_actual_pred_best["ACTUAL_PRODUCTION"],
        df_actual_pred_best["PREDICTED_PRODUCTION"]
    )
)
mae_ton_pooled = mean_absolute_error(
    df_actual_pred_best["ACTUAL_PRODUCTION"],
    df_actual_pred_best["PREDICTED_PRODUCTION"]
)

print("=" * 55)
print("POOLED METRICS PADA SKENARIO TERBAIK (SW-8):")
print("=" * 55)
print(f"[1] PRODUKTIVITAS (Target Model Sebenarnya):")
print(f"    R²   : {r2_prodv_pooled:.4f}")
print(f"    RMSE : {rmse_prodv_pooled:.4f} ton/ha")
print(f"    MAE  : {mae_prodv_pooled:.4f} ton/ha")
print(f"    MAPE : {mape_prodv_pooled:.2f}%")
print(f"\n[2] PRODUKSI (Hasil Konversi Luas Panen):")
print(f"    R²   : {r2_ton_pooled:.4f}")
print(f"    RMSE : {rmse_ton_pooled:,.2f} ton")
print(f"    MAE  : {mae_ton_pooled:,.2f} ton")
print(f"    Jumlah titik sampel uji: {len(df_actual_pred_best)}")
print("=" * 55)

# ---------------------------------------------------------
# 5. Scatter Plot Side-by-Side (Produktivitas & Produksi)
# ---------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))

# Plot 1: Produktivitas (ton/ha)
ax1.scatter(
    df_actual_pred_best["ACTUAL_PRODUCTIVITY"],
    df_actual_pred_best["PREDICTED_PRODUCTIVITY"],
    s=40,
    color="#1f77b4",
    alpha=0.75,
    edgecolors="black",
    linewidths=0.4
)
min_v = min(df_actual_pred_best["ACTUAL_PRODUCTIVITY"].min(), df_actual_pred_best["PREDICTED_PRODUCTIVITY"].min()) * 0.98
max_v = max(df_actual_pred_best["ACTUAL_PRODUCTIVITY"].max(), df_actual_pred_best["PREDICTED_PRODUCTIVITY"].max()) * 1.02
ax1.plot([min_v, max_v], [min_v, max_v], linestyle="--", linewidth=1.2, color="red", label="Ideal (y = x)")
ax1.set_xlabel("Actual Productivity (ton/ha)", fontsize=10, fontweight="bold")
ax1.set_ylabel("Predicted Productivity (ton/ha)", fontsize=10, fontweight="bold")
ax1.set_title(f"Actual vs Predicted Productivity\n(R² = {r2_prodv_pooled:.4f}, RMSE = {rmse_prodv_pooled:.4f} ton/ha)", fontsize=11)
ax1.legend(frameon=True, fontsize=8.5)
ax1.grid(alpha=0.25, linewidth=0.5)

# Plot 2: Produksi (ton)
ax2.scatter(
    df_actual_pred_best["ACTUAL_PRODUCTION"],
    df_actual_pred_best["PREDICTED_PRODUCTION"],
    s=40,
    color="#2ca02c",
    alpha=0.75,
    edgecolors="black",
    linewidths=0.4
)
min_p = min(df_actual_pred_best["ACTUAL_PRODUCTION"].min(), df_actual_pred_best["PREDICTED_PRODUCTION"].min()) * 0.95
max_p = max(df_actual_pred_best["ACTUAL_PRODUCTION"].max(), df_actual_pred_best["PREDICTED_PRODUCTION"].max()) * 1.05
ax2.plot([min_p, max_p], [min_p, max_p], linestyle="--", linewidth=1.2, color="red", label="Ideal (y = x)")
ax2.set_xlabel("Actual Production (ton)", fontsize=10, fontweight="bold")
ax2.set_ylabel("Predicted Production (ton)", fontsize=10, fontweight="bold")
ax2.set_title(f"Actual vs Predicted Production (Konversi)\n(R² = {r2_ton_pooled:.4f}, RMSE = {rmse_ton_pooled:,.2f} ton)", fontsize=11)
ax2.legend(frameon=True, fontsize=8.5)
ax2.grid(alpha=0.25, linewidth=0.5)

plt.tight_layout()
plt.savefig("actual_vs_predicted_sw_best.png", dpi=300, bbox_inches="tight")
plt.show()

print("Gambar tersimpan: actual_vs_predicted_sw_best.png")


Membangun Actual vs. Predicted untuk skenario terbaik...
Best window: 7 tahun


POOLED METRICS PADA SKENARIO TERBAIK (SW-8):
[1] PRODUKTIVITAS (Target Model Sebenarnya):
    R²   : 0.3613
    RMSE : 0.3306 ton/ha
    MAE  : 0.2372 ton/ha
    MAPE : 4.07%

[2] PRODUKSI (Hasil Konversi Luas Panen):
    R²   : 0.9840
    RMSE : 886.30 ton
    MAE  : 572.30 ton
    Jumlah titik sampel uji: 81


Gambar tersimpan: actual_vs_predicted_sw_best.png


C:\Users\acer\AppData\Local\Temp\ipykernel_16572\1840661241.py:241: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
